In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="2"


In [2]:
import datasets
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
import copy

In [3]:
from transformers import AutoTokenizer

# Build ManyTypes4Py

In [68]:
path = "../../data/ManyTypes4PyDataset-v0.7/"

In [69]:
repos = os.listdir(os.path.join(path, "processed_projects_clean"))
repos = [el for el in repos if not(el.startswith("."))]

In [70]:
# train and val splits
df = pd.read_csv(path + "dataset_split.csv")

In [71]:
# TODO:
# 1. split dataset into train and val by repos
# each json file contains one repo with variables, functions (with variables), classes (with variables) and code
# for now, use only variables from main code
# 2. build first split with 2k len by one or more file
# 3. pass

In [72]:
from sklearn.model_selection import train_test_split

train_repos, val_repos = train_test_split(repos, test_size=0.2, random_state=42)
test_repos, val_repos = train_test_split(val_repos, test_size=0.5, random_state=42)

In [73]:
def check_variables(code, variables):
    new_variables = {}
    for variable in variables.keys():
        if variable in code:
            new_variables[variable] = variables[variable]
    return new_variables

In [74]:
# process one repo
def extract_repo_data(path, repo):
    repo_path = os.path.join(path, "processed_projects_clean", repo)
    # load repo code
    with open(repo_path, "rb") as f:
        repo_code = json.load(f)
    repo_name = list(repo_code.keys())[0] # check if all repos have one key
    repo_files = list(repo_code[repo_name]['src_files'].keys())
    repo_data = {
        "repo_name": [],
        "fname": [],
        "code": [],
        "variables": [],
    }
    for fl in repo_files:
        # some code files can be empty
        if len(repo_code[repo_name]['src_files'][fl]["untyped_seq"]) != 0:
            repo_data["repo_name"].append(repo_name)
            repo_data["fname"].append(fl)
            repo_data["code"].append(repo_code[repo_name]['src_files'][fl]["untyped_seq"].replace("[EOL] ", "\n").replace("[EOL]", "\n"))
            # add check if all variables are in the code
            new_variables = check_variables(repo_code[repo_name]['src_files'][fl]["untyped_seq"],
                                            repo_code[repo_name]['src_files'][fl]["variables"])
            repo_data["variables"].append(new_variables)
            # add variables from classes and fucntions
            # TODO: add for each variable class and function name (if applicable)
            if len(repo_code[repo_name]['src_files'][fl]["classes"]) != 0:
                # look for class variables and add them
                for el in repo_code[repo_name]['src_files'][fl]["classes"]:
                    cls_variables = check_variables(repo_code[repo_name]['src_files'][fl]["untyped_seq"],
                                                    el["variables"])
                    repo_data["variables"][-1].update(cls_variables)
            if len(repo_code[repo_name]['src_files'][fl]["funcs"]) != 0:
                # look for class variables and add them
                for el in repo_code[repo_name]['src_files'][fl]["funcs"]:
                    cls_variables = check_variables(repo_code[repo_name]['src_files'][fl]["untyped_seq"],
                                                    el["variables"])
                    repo_data["variables"][-1].update(cls_variables)
    return repo_data



In [61]:
# build repo-file level dataset
file_level_data = []
for repo in tqdm(train_repos):
    res = extract_repo_data(path, repo)
    file_level_data.append(res)

  4%|▍         | 168/4073 [00:02<00:41, 93.24it/s]Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f8c6c670d00>>
Traceback (most recent call last):
  File "/home/jovyan/gkuzmin/rmt_it/conda_rmt_it_venv/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 
100%|██████████| 4073/4073 [00:52<00:00, 77.87it/s] 


In [45]:
file_level_data[0].keys()

dict_keys(['repo_name', 'fname', 'code', 'variables'])

In [46]:
# concat all file-level data into one dataset
train_ds_dict = copy.deepcopy(file_level_data[0])
for el in tqdm(file_level_data[1:]):
    #for key in train_ds_dict.keys():
    #    train_ds_dict[key].append(el[key])
    train_ds_dict = {key: train_ds_dict[key] + el[key] for key in train_ds_dict.keys()}
# recast variables into tuples for compatibility with hf datasets
train_ds_dict["variables"] = [[(key, value) for key, value in el.items()] for el in train_ds_dict["variables"]]

100%|██████████| 4072/4072 [00:10<00:00, 382.12it/s] 


In [48]:
train_ds = datasets.Dataset.from_dict(train_ds_dict)

In [75]:
# build repo-file level dataset
repo_names = [train_repos, val_repos, test_repos]
repo_keys = ["train", "validation", "test"]
proc_data = {}
for repo_split, repo_key in zip(repo_names, repo_keys):
    print(repo_key)
    file_level_data = []
    for repo in tqdm(repo_split):
        res = extract_repo_data(path, repo)
        file_level_data.append(res)
    split_ds_dict = copy.deepcopy(file_level_data[0])
    for el in tqdm(file_level_data[1:]):
        #for key in train_ds_dict.keys():
        #    train_ds_dict[key].append(el[key])
        split_ds_dict = {key: split_ds_dict[key] + el[key] for key in split_ds_dict.keys()}
    split_ds_dict["variables"] = [[(key, value) for key, value in el.items()] for el in split_ds_dict["variables"]]
    proc_data[repo_key] = datasets.Dataset.from_dict(split_ds_dict)

train


100%|██████████| 4072/4072 [00:11<00:00, 356.08it/s] 


validation


100%|██████████| 509/509 [00:00<00:00, 2285.05it/s]


test


100%|██████████| 508/508 [00:00<00:00, 4728.21it/s]


In [76]:
manytypes_ds = datasets.DatasetDict(proc_data)
manytypes_ds.save_to_disk("../data/manytypes_raw")

Saving the dataset (0/1 shards):   0%|          | 0/141525 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/18576 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/12630 [00:00<?, ? examples/s]

### Load raw dataset, filter, rebuild

In [94]:
# now prepare prompts and filter by length
manytypes_ds = datasets.load_from_disk("../data/manytypes_raw")

In [4]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

In [5]:
def caclulate_length(example):
    messages = [
        {"role": "user", "content": example["code"]}
    ]
    length = len(tokenizer.apply_chat_template(messages, tokenize=True))
    example["length"] = length
    return example

In [97]:
manytypes_ds = manytypes_ds.map(caclulate_length)

Map:   0%|          | 0/12630 [00:00<?, ? examples/s]

In [98]:
tr_lens = manytypes_ds["train"]["length"]
np.mean(tr_lens), np.percentile(tr_lens, 10), np.percentile(tr_lens, 20), np.percentile(tr_lens, 80), np.percentile(tr_lens, 95)

(np.float64(1020.0314856032503),
 np.float64(41.0),
 np.float64(72.0),
 np.float64(997.0),
 np.float64(3262.0))

In [99]:
def stack_one_repo(repo_1, min_len=0, max_len=2048):
    curr_thr = np.random.randint(min_len, max_len)
    # after choose samples up to curr_thr and concatenate them
    idx = 0
    new_repo_1 = {
        "repo_name": [],
        "fname": [],
        "code": [],
        "variables": [],
        "length": [],
    }
    while idx < len(repo_1):
        curr_len = repo_1[idx]["length"]
        curr_repo = repo_1[idx]["repo_name"]
        curr_fnames = [repo_1[idx]["fname"]]
        curr_code = repo_1[idx]["code"]
        curr_vars = repo_1[idx]["variables"]
        while curr_len < curr_thr and idx+1 < len(repo_1):
            # iterate over ids to concate text
            idx += 1
            curr_len += repo_1[idx]["length"]
            curr_fnames += [repo_1[idx]["fname"]]
            curr_code += "\n" + repo_1[idx]["code"]
            curr_vars += repo_1[idx]["variables"]
        idx += 1
        new_repo_1["repo_name"].append(curr_repo)
        new_repo_1["fname"].append(curr_fnames)
        new_repo_1["code"].append(curr_code)
        new_repo_1["variables"].append(curr_vars)
        new_repo_1["length"].append(curr_len)
    return datasets.Dataset.from_dict(new_repo_1)

In [100]:
all_repos = {}
for split in manytypes_ds.keys():
    pd_train = manytypes_ds[split].to_pandas()#["repo_name"]
    grouped_ids = pd_train.groupby("repo_name")

    filtered_repos = []
    new_repos = []

    for unique_repo_ids in tqdm(grouped_ids.groups.values()):
        repo_1 = manytypes_ds[split].select(unique_repo_ids)
        filtered_repos.append(repo_1)
    new_repo_1 = stack_one_repo(filtered_repos[0], 0, 2048)
    vars_col_type = new_repo_1.features["variables"]
    datasets.disable_progress_bars()
    for f_repo in tqdm(filtered_repos):
        new_repo_1 = stack_one_repo(f_repo, 0, 2048)
        new_repos.append(new_repo_1.cast_column("variables", vars_col_type))
    datasets.enable_progress_bars()
    all_repos[split] = datasets.concatenate_datasets(new_repos)


100%|██████████| 505/505 [00:11<00:00, 43.23it/s]


In [101]:
#code_repos_2k = datasets.concatenate_datasets(new_repos)

In [102]:
code_repos_2k = datasets.DatasetDict(all_repos)

In [103]:
def flatten_samples(example):
    prompt_template = lambda variable: f"What is the type of variable {variable}?"
    new_examples = {key: [] for key in example.keys()}#copy.deepcopy(example)
    new_examples["answer"] = []
    for v in example["variables"]:
        for key in new_examples.keys():
            if key == "variables":
                new_examples[key].append(prompt_template(v[0]))
                new_examples["answer"].append(v[1])
            elif key == "answer":
                pass
            else:
                new_examples[key].append(example[key])
    return new_examples

In [104]:
# filter samples with empty variables
code_repos_2k = code_repos_2k.filter(lambda example: (len(example["variables"]) > 0) and not(all([el[1] == '' for el in example["variables"]])))

Filter:   0%|          | 0/56927 [00:00<?, ? examples/s]

Filter:   0%|          | 0/6982 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5182 [00:00<?, ? examples/s]

In [139]:
# generate prompts and answers, and recalc length
multi_code_repos_2k = code_repos_2k.map(flatten_samples)
flatten_dataset_2k = {}
for split in multi_code_repos_2k.keys():
    flat_code_repos_2k = {key: [] for key in multi_code_repos_2k[split][0].keys()}
    for el in tqdm(multi_code_repos_2k[split]):
        for key in flat_code_repos_2k.keys():
            flat_code_repos_2k[key] += el[key]
    fl_ds = datasets.Dataset.from_dict(flat_code_repos_2k)
    flatten_dataset_2k[split] = fl_ds

Map:   0%|          | 0/31548 [00:00<?, ? examples/s]

Map:   0%|          | 0/4327 [00:00<?, ? examples/s]

Map:   0%|          | 0/3274 [00:00<?, ? examples/s]

100%|██████████| 3274/3274 [00:00<00:00, 4963.25it/s]


In [145]:
# recalc len based on chat template and filter by len
def prepare_text(example):
    messages = [
        {"role": "user", "content": f"Question:\n{example['question']}\nContext:\n{example['context']}\n"},
        {"role": "assistant", "content": f"Answer:\n{example['answer']}\n"},
    ]
    ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )
    example["full_len"] = len(ids)
    return example


def tokenize_text(example):
    ids = tokenizer(example["full_text"])["input_ids"]
    example["full_len"] = len(ids)
    return example

In [141]:
flatten_dataset_2k = datasets.DatasetDict(flatten_dataset_2k)
flatten_dataset_2k = flatten_dataset_2k.rename_column("code", "context")
flatten_dataset_2k = flatten_dataset_2k.rename_column("variables", "question")

#flatten_dataset_2k = flatten_dataset_2k.map(prepare_text)

In [146]:
flatten_dataset_2k = flatten_dataset_2k.map(prepare_text, batched=False)

Map:   0%|          | 0/232994 [00:00<?, ? examples/s]

Map:   0%|          | 0/35886 [00:00<?, ? examples/s]

Map:   0%|          | 0/23529 [00:00<?, ? examples/s]

In [147]:
#flatten_dataset_2k_prepared = flatten_dataset_2k.map(tokenize_text)
#tokenizer(flatten_dataset_2k["train"][0]["full_text"])["input_ids"]

In [148]:
flatten_dataset_2k.save_to_disk("../data/manytypes_flat_stacked_0k_2k")

Saving the dataset (0/5 shards):   0%|          | 0/232994 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/35886 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/23529 [00:00<?, ? examples/s]

In [150]:
flatten_dataset_2k_filtered = flatten_dataset_2k.filter(lambda example: example["full_len"] < 1920)

Filter:   0%|          | 0/232994 [00:00<?, ? examples/s]

Filter:   0%|          | 0/35886 [00:00<?, ? examples/s]

Filter:   0%|          | 0/23529 [00:00<?, ? examples/s]

In [151]:
flatten_dataset_2k_filtered.save_to_disk("../data/manytypes_flat_stacked_0k_2k")

Saving the dataset (0/3 shards):   0%|          | 0/136284 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/14918 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/14697 [00:00<?, ? examples/s]

In [ ]:
# now tokenize and calc length

In [ ]:
flatten_dataset_2k.save_to_disk("../data/manytypes_flat_stacked_0k_2k")

In [152]:
flatten_dataset_2k_filtered_empty = flatten_dataset_2k_filtered.filter(lambda example: example["answer"] != "")

Filter:   0%|          | 0/136284 [00:00<?, ? examples/s]

Filter:   0%|          | 0/14918 [00:00<?, ? examples/s]

Filter:   0%|          | 0/14697 [00:00<?, ? examples/s]

In [154]:
flatten_dataset_2k_filtered_empty.save_to_disk("../data/manytypes_flat_stacked_no_empty_0k_2k")

Saving the dataset (0/2 shards):   0%|          | 0/112079 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/12510 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/11973 [00:00<?, ? examples/s]

In [63]:
flatten_dataset_2k_filtered_empty = datasets.load_from_disk("../data/manytypes_flat_stacked_no_empty_0k_2k")


In [64]:
flatten_dataset_2k_filtered_empty

DatasetDict({
    train: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 112079
    })
    validation: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 12510
    })
    test: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 11973
    })
})

In [66]:
# subsample validation to 10% to speed up training
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(flatten_dataset_2k_filtered_empty["validation"])),
                           int(0.1 * len(flatten_dataset_2k_filtered_empty["validation"])),
                           replace=False)
flatten_dataset_2k_filtered_empty_subsample = copy.deepcopy(flatten_dataset_2k_filtered_empty)
flatten_dataset_2k_filtered_empty_subsample["validation"] = flatten_dataset_2k_filtered_empty["validation"].select(val_ids)

In [68]:
flatten_dataset_2k_filtered_empty_subsample.save_to_disk("../../data/mt_prepared/manytypes_flat_stacked_no_empty_sv_0k_2k")

Saving the dataset (0/1 shards):   0%|          | 0/112079 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1251 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/11973 [00:00<?, ? examples/s]

In [39]:
flatten_dataset_2k_filtered_empty.save_to_disk("../../data/mt_prepared/manytypes_flat_stacked_no_empty_0k_2k")

Saving the dataset (0/1 shards):   0%|          | 0/112079 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/12510 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/11973 [00:00<?, ? examples/s]

In [14]:
flatten_dataset_2k_filtered_empty["validation"]["answer"][0]

'builtins.str'

In [15]:
test_answs = flatten_dataset_2k_filtered_empty["validation"]["answer"]

In [17]:
unique_answs = np.unique(test_answs, return_counts=True)

In [37]:
test_answs_ids = np.argsort(unique_answs[1])[::-1][200:]
unique_answs[1][test_answs_ids], unique_answs[0][test_answs_ids]

(array([3, 3, 3, ..., 1, 1, 1], shape=(1304,)),
 array(['typing.Optional[builtins.bytes]',
        'typing.List[typing.Union[typing.Dict[builtins.str, builtins.str], typing.Tuple[builtins.str, builtins.str, builtins.int, builtins.bool]]]',
        'wemake_python_styleguide.visitors.ast.loops.WrongComprehensionVisitor',
        ..., 'UserStatus', 'aiko.utils.ProxyAttr',
        'aiko.application.Application'], shape=(1304,), dtype='<U661'))

### Manytypes on 512 length

In [4]:
# now prepare prompts and filter by length
manytypes_ds = datasets.load_from_disk("../data/manytypes_raw")

In [5]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

In [6]:
def caclulate_length(example):
    messages = [
        {"role": "user", "content": example["code"]}
    ]
    length = len(tokenizer.apply_chat_template(messages, tokenize=True))
    example["length"] = length
    return example

In [7]:
manytypes_ds = manytypes_ds.map(caclulate_length)

In [8]:
def stack_one_repo(repo_1, min_len=0, max_len=2048):
    curr_thr = np.random.randint(min_len, max_len)
    # after choose samples up to curr_thr and concatenate them
    idx = 0
    new_repo_1 = {
        "repo_name": [],
        "fname": [],
        "code": [],
        "variables": [],
        "length": [],
    }
    while idx < len(repo_1):
        curr_len = repo_1[idx]["length"]
        curr_repo = repo_1[idx]["repo_name"]
        curr_fnames = [repo_1[idx]["fname"]]
        curr_code = repo_1[idx]["code"]
        curr_vars = repo_1[idx]["variables"]
        while curr_len < curr_thr and idx+1 < len(repo_1):
            # iterate over ids to concate text
            idx += 1
            curr_len += repo_1[idx]["length"]
            curr_fnames += [repo_1[idx]["fname"]]
            curr_code += "\n" + repo_1[idx]["code"]
            curr_vars += repo_1[idx]["variables"]
        idx += 1
        new_repo_1["repo_name"].append(curr_repo)
        new_repo_1["fname"].append(curr_fnames)
        new_repo_1["code"].append(curr_code)
        new_repo_1["variables"].append(curr_vars)
        new_repo_1["length"].append(curr_len)
    return datasets.Dataset.from_dict(new_repo_1)

In [9]:
all_repos = {}
min_len = 0
max_len = 512
split_suffix = "512"
for split in manytypes_ds.keys():
    pd_train = manytypes_ds[split].to_pandas()#["repo_name"]
    grouped_ids = pd_train.groupby("repo_name")

    filtered_repos = []
    new_repos = []

    for unique_repo_ids in tqdm(grouped_ids.groups.values()):
        repo_1 = manytypes_ds[split].select(unique_repo_ids)
        filtered_repos.append(repo_1)
    new_repo_1 = stack_one_repo(filtered_repos[0], min_len, max_len)
    vars_col_type = new_repo_1.features["variables"]
    datasets.disable_progress_bars()
    for f_repo in tqdm(filtered_repos):
        new_repo_1 = stack_one_repo(f_repo, min_len, max_len)
        new_repos.append(new_repo_1.cast_column("variables", vars_col_type))
    datasets.enable_progress_bars()
    all_repos[split] = datasets.concatenate_datasets(new_repos)


100%|██████████| 505/505 [00:12<00:00, 42.02it/s]


In [10]:
#code_repos_2k = datasets.concatenate_datasets(new_repos)

In [11]:
code_repos_4k = datasets.DatasetDict(all_repos)

In [12]:
def flatten_samples(example):
    prompt_template = lambda variable: f"What is the type of variable {variable}?"
    new_examples = {key: [] for key in example.keys()}#copy.deepcopy(example)
    new_examples["answer"] = []
    for v in example["variables"]:
        for key in new_examples.keys():
            if key == "variables":
                new_examples[key].append(prompt_template(v[0]))
                new_examples["answer"].append(v[1])
            elif key == "answer":
                pass
            else:
                new_examples[key].append(example[key])
    return new_examples

In [13]:
# filter samples with empty variables
code_repos_4k = code_repos_4k.filter(lambda example: (len(example["variables"]) > 0) and not(all([el[1] == '' for el in example["variables"]])))

Filter:   0%|          | 0/93591 [00:00<?, ? examples/s]

Filter:   0%|          | 0/12020 [00:00<?, ? examples/s]

Filter:   0%|          | 0/8307 [00:00<?, ? examples/s]

In [14]:
# generate prompts and answers, and recalc length
multi_code_repos_2k = code_repos_4k.map(flatten_samples)
flatten_dataset_2k = {}
for split in multi_code_repos_2k.keys():
    flat_code_repos_2k = {key: [] for key in multi_code_repos_2k[split][0].keys()}
    for el in tqdm(multi_code_repos_2k[split]):
        for key in flat_code_repos_2k.keys():
            flat_code_repos_2k[key] += el[key]
    fl_ds = datasets.Dataset.from_dict(flat_code_repos_2k)
    flatten_dataset_2k[split] = fl_ds

Map:   0%|          | 0/45204 [00:00<?, ? examples/s]

Map:   0%|          | 0/6635 [00:00<?, ? examples/s]

Map:   0%|          | 0/4782 [00:00<?, ? examples/s]

100%|██████████| 4782/4782 [00:00<00:00, 8169.32it/s]


In [15]:
# recalc len based on chat template and filter by len
def prepare_text(example):
    messages = [
        {"role": "user", "content": f"Question:\n{example['question']}\nContext:\n{example['context']}\n"},
        {"role": "assistant", "content": f"Answer:\n{example['answer']}\n"},
    ]
    ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )
    example["full_len"] = len(ids)
    return example


def tokenize_text(example):
    ids = tokenizer(example["full_text"])["input_ids"]
    example["full_len"] = len(ids)
    return example

In [16]:
flatten_dataset_2k = datasets.DatasetDict(flatten_dataset_2k)
flatten_dataset_2k = flatten_dataset_2k.rename_column("code", "context")
flatten_dataset_2k = flatten_dataset_2k.rename_column("variables", "question")

#flatten_dataset_2k = flatten_dataset_2k.map(prepare_text)

In [17]:
flatten_dataset_2k = flatten_dataset_2k.map(prepare_text, batched=False)

Map:   0%|          | 0/226440 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (140522 > 131072). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/34942 [00:00<?, ? examples/s]

Map:   0%|          | 0/22840 [00:00<?, ? examples/s]

In [18]:
#flatten_dataset_2k_prepared = flatten_dataset_2k.map(tokenize_text)
#tokenizer(flatten_dataset_2k["train"][0]["full_text"])["input_ids"]

In [19]:
flatten_dataset_2k.save_to_disk(f"../../data/mt_intermediate/manytypes_flat_stacked_0k_{split_suffix}")

Saving the dataset (0/4 shards):   0%|          | 0/226440 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/34942 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/22840 [00:00<?, ? examples/s]

In [20]:
flatten_dataset_2k_filtered = flatten_dataset_2k.filter(lambda example: example["full_len"] < max_len-64)

Filter:   0%|          | 0/226440 [00:00<?, ? examples/s]

Filter:   0%|          | 0/34942 [00:00<?, ? examples/s]

Filter:   0%|          | 0/22840 [00:00<?, ? examples/s]

In [21]:
flatten_dataset_2k_filtered.save_to_disk(f"../../data/mt_intermediate/manytypes_flat_stacked_0k_{split_suffix}")

Saving the dataset (0/1 shards):   0%|          | 0/39646 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5278 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4854 [00:00<?, ? examples/s]

In [22]:
# now tokenize and calc length

In [23]:
flatten_dataset_2k.save_to_disk(f"../../data/mt_intermediate/manytypes_flat_stacked_0k_{split_suffix}")

Saving the dataset (0/4 shards):   0%|          | 0/226440 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/34942 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/22840 [00:00<?, ? examples/s]

In [24]:
flatten_dataset_2k_filtered_empty = flatten_dataset_2k_filtered.filter(lambda example: example["answer"] != "")

Filter:   0%|          | 0/39646 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5278 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4854 [00:00<?, ? examples/s]

In [25]:
flatten_dataset_2k_filtered_empty.save_to_disk(f"../../data/mt_prepared/manytypes_flat_stacked_no_empty_0k_{split_suffix}")

Saving the dataset (0/1 shards):   0%|          | 0/35224 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4708 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4269 [00:00<?, ? examples/s]

In [26]:
flatten_dataset_2k_filtered_empty = datasets.load_from_disk(f"../../data/mt_prepared/manytypes_flat_stacked_no_empty_0k_{split_suffix}")


In [27]:
flatten_dataset_2k_filtered_empty

DatasetDict({
    train: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 35224
    })
    validation: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 4708
    })
    test: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 4269
    })
})

In [28]:
# subsample validation to 10% to speed up training
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(flatten_dataset_2k_filtered_empty["validation"])),
                           int(0.1 * len(flatten_dataset_2k_filtered_empty["validation"])),
                           replace=False)
flatten_dataset_2k_filtered_empty_subsample = copy.deepcopy(flatten_dataset_2k_filtered_empty)
flatten_dataset_2k_filtered_empty_subsample["validation"] = flatten_dataset_2k_filtered_empty["validation"].select(val_ids)

In [29]:
flatten_dataset_2k_filtered_empty_subsample.save_to_disk(f"../../data/mt_prepared/manytypes_flat_stacked_no_empty_sv_0k_{split_suffix}")

Saving the dataset (0/1 shards):   0%|          | 0/35224 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/470 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4269 [00:00<?, ? examples/s]

### Manytypes on 1k length

In [6]:
# now prepare prompts and filter by length
manytypes_ds = datasets.load_from_disk("../data/manytypes_raw")

In [7]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

In [8]:
def caclulate_length(example):
    messages = [
        {"role": "user", "content": example["code"]}
    ]
    length = len(tokenizer.apply_chat_template(messages, tokenize=True))
    example["length"] = length
    return example

In [9]:
manytypes_ds = manytypes_ds.map(caclulate_length)

In [10]:
def stack_one_repo(repo_1, min_len=0, max_len=2048):
    curr_thr = np.random.randint(min_len, max_len)
    # after choose samples up to curr_thr and concatenate them
    idx = 0
    new_repo_1 = {
        "repo_name": [],
        "fname": [],
        "code": [],
        "variables": [],
        "length": [],
    }
    while idx < len(repo_1):
        curr_len = repo_1[idx]["length"]
        curr_repo = repo_1[idx]["repo_name"]
        curr_fnames = [repo_1[idx]["fname"]]
        curr_code = repo_1[idx]["code"]
        curr_vars = repo_1[idx]["variables"]
        while curr_len < curr_thr and idx+1 < len(repo_1):
            # iterate over ids to concate text
            idx += 1
            curr_len += repo_1[idx]["length"]
            curr_fnames += [repo_1[idx]["fname"]]
            curr_code += "\n" + repo_1[idx]["code"]
            curr_vars += repo_1[idx]["variables"]
        idx += 1
        new_repo_1["repo_name"].append(curr_repo)
        new_repo_1["fname"].append(curr_fnames)
        new_repo_1["code"].append(curr_code)
        new_repo_1["variables"].append(curr_vars)
        new_repo_1["length"].append(curr_len)
    return datasets.Dataset.from_dict(new_repo_1)

In [11]:
all_repos = {}
min_len = 0
max_len = 1024
for split in manytypes_ds.keys():
    pd_train = manytypes_ds[split].to_pandas()#["repo_name"]
    grouped_ids = pd_train.groupby("repo_name")

    filtered_repos = []
    new_repos = []

    for unique_repo_ids in tqdm(grouped_ids.groups.values()):
        repo_1 = manytypes_ds[split].select(unique_repo_ids)
        filtered_repos.append(repo_1)
    new_repo_1 = stack_one_repo(filtered_repos[0], min_len, max_len)
    vars_col_type = new_repo_1.features["variables"]
    datasets.disable_progress_bars()
    for f_repo in tqdm(filtered_repos):
        new_repo_1 = stack_one_repo(f_repo, min_len, max_len)
        new_repos.append(new_repo_1.cast_column("variables", vars_col_type))
    datasets.enable_progress_bars()
    all_repos[split] = datasets.concatenate_datasets(new_repos)


100%|██████████| 505/505 [00:11<00:00, 43.34it/s]


In [12]:
#code_repos_2k = datasets.concatenate_datasets(new_repos)

In [13]:
code_repos_4k = datasets.DatasetDict(all_repos)

In [14]:
def flatten_samples(example):
    prompt_template = lambda variable: f"What is the type of variable {variable}?"
    new_examples = {key: [] for key in example.keys()}#copy.deepcopy(example)
    new_examples["answer"] = []
    for v in example["variables"]:
        for key in new_examples.keys():
            if key == "variables":
                new_examples[key].append(prompt_template(v[0]))
                new_examples["answer"].append(v[1])
            elif key == "answer":
                pass
            else:
                new_examples[key].append(example[key])
    return new_examples

In [15]:
# filter samples with empty variables
code_repos_4k = code_repos_4k.filter(lambda example: (len(example["variables"]) > 0) and not(all([el[1] == '' for el in example["variables"]])))

Filter:   0%|          | 0/75438 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9401 [00:00<?, ? examples/s]

Filter:   0%|          | 0/6584 [00:00<?, ? examples/s]

In [16]:
# generate prompts and answers, and recalc length
multi_code_repos_2k = code_repos_4k.map(flatten_samples)
flatten_dataset_2k = {}
for split in multi_code_repos_2k.keys():
    flat_code_repos_2k = {key: [] for key in multi_code_repos_2k[split][0].keys()}
    for el in tqdm(multi_code_repos_2k[split]):
        for key in flat_code_repos_2k.keys():
            flat_code_repos_2k[key] += el[key]
    fl_ds = datasets.Dataset.from_dict(flat_code_repos_2k)
    flatten_dataset_2k[split] = fl_ds

Map:   0%|          | 0/37973 [00:00<?, ? examples/s]

Map:   0%|          | 0/5567 [00:00<?, ? examples/s]

Map:   0%|          | 0/4096 [00:00<?, ? examples/s]

100%|██████████| 4096/4096 [00:00<00:00, 7241.91it/s]


In [17]:
# recalc len based on chat template and filter by len
def prepare_text(example):
    messages = [
        {"role": "user", "content": f"Question:\n{example['question']}\nContext:\n{example['context']}\n"},
        {"role": "assistant", "content": f"Answer:\n{example['answer']}\n"},
    ]
    ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )
    example["full_len"] = len(ids)
    return example


def tokenize_text(example):
    ids = tokenizer(example["full_text"])["input_ids"]
    example["full_len"] = len(ids)
    return example

In [18]:
flatten_dataset_2k = datasets.DatasetDict(flatten_dataset_2k)
flatten_dataset_2k = flatten_dataset_2k.rename_column("code", "context")
flatten_dataset_2k = flatten_dataset_2k.rename_column("variables", "question")

#flatten_dataset_2k = flatten_dataset_2k.map(prepare_text)

In [19]:
flatten_dataset_2k = flatten_dataset_2k.map(prepare_text, batched=False)

Map:   0%|          | 0/230145 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (140522 > 131072). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/35233 [00:00<?, ? examples/s]

Map:   0%|          | 0/23292 [00:00<?, ? examples/s]

In [20]:
#flatten_dataset_2k_prepared = flatten_dataset_2k.map(tokenize_text)
#tokenizer(flatten_dataset_2k["train"][0]["full_text"])["input_ids"]

In [21]:
flatten_dataset_2k.save_to_disk("../../data/mt_intermediate/manytypes_flat_stacked_0k_1k")

Saving the dataset (0/4 shards):   0%|          | 0/230145 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/35233 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/23292 [00:00<?, ? examples/s]

In [22]:
flatten_dataset_2k_filtered = flatten_dataset_2k.filter(lambda example: example["full_len"] < 960)

Filter:   0%|          | 0/230145 [00:00<?, ? examples/s]

Filter:   0%|          | 0/35233 [00:00<?, ? examples/s]

Filter:   0%|          | 0/23292 [00:00<?, ? examples/s]

In [23]:
flatten_dataset_2k_filtered.save_to_disk("../../data/mt_intermediate/manytypes_flat_stacked_0k_1k")

Saving the dataset (0/2 shards):   0%|          | 0/84382 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/11270 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/9737 [00:00<?, ? examples/s]

In [24]:
# now tokenize and calc length

In [25]:
flatten_dataset_2k.save_to_disk("../../data/mt_intermediate/manytypes_flat_stacked_0k_1k")

Saving the dataset (0/4 shards):   0%|          | 0/230145 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/35233 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/23292 [00:00<?, ? examples/s]

In [26]:
flatten_dataset_2k_filtered_empty = flatten_dataset_2k_filtered.filter(lambda example: example["answer"] != "")

Filter:   0%|          | 0/84382 [00:00<?, ? examples/s]

Filter:   0%|          | 0/11270 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9737 [00:00<?, ? examples/s]

In [27]:
flatten_dataset_2k_filtered_empty.save_to_disk("../../data/mt_prepared/manytypes_flat_stacked_no_empty_0k_1k")

Saving the dataset (0/2 shards):   0%|          | 0/71629 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/9654 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/8187 [00:00<?, ? examples/s]

In [28]:
flatten_dataset_2k_filtered_empty = datasets.load_from_disk("../../data/mt_prepared/manytypes_flat_stacked_no_empty_0k_1k")


In [29]:
flatten_dataset_2k_filtered_empty

DatasetDict({
    train: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 71629
    })
    validation: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 9654
    })
    test: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 8187
    })
})

In [30]:
# subsample validation to 10% to speed up training
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(flatten_dataset_2k_filtered_empty["validation"])),
                           int(0.1 * len(flatten_dataset_2k_filtered_empty["validation"])),
                           replace=False)
flatten_dataset_2k_filtered_empty_subsample = copy.deepcopy(flatten_dataset_2k_filtered_empty)
flatten_dataset_2k_filtered_empty_subsample["validation"] = flatten_dataset_2k_filtered_empty["validation"].select(val_ids)

In [31]:
flatten_dataset_2k_filtered_empty_subsample.save_to_disk("../../data/mt_prepared/manytypes_flat_stacked_no_empty_sv_0k_1k")

Saving the dataset (0/1 shards):   0%|          | 0/71629 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/965 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/8187 [00:00<?, ? examples/s]

### Manytypes on 4k length

In [40]:
# now prepare prompts and filter by length
manytypes_ds = datasets.load_from_disk("../data/manytypes_raw")

In [41]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

In [42]:
def caclulate_length(example):
    messages = [
        {"role": "user", "content": example["code"]}
    ]
    length = len(tokenizer.apply_chat_template(messages, tokenize=True))
    example["length"] = length
    return example

In [43]:
manytypes_ds = manytypes_ds.map(caclulate_length)

In [44]:
def stack_one_repo(repo_1, min_len=0, max_len=2048):
    curr_thr = np.random.randint(min_len, max_len)
    # after choose samples up to curr_thr and concatenate them
    idx = 0
    new_repo_1 = {
        "repo_name": [],
        "fname": [],
        "code": [],
        "variables": [],
        "length": [],
    }
    while idx < len(repo_1):
        curr_len = repo_1[idx]["length"]
        curr_repo = repo_1[idx]["repo_name"]
        curr_fnames = [repo_1[idx]["fname"]]
        curr_code = repo_1[idx]["code"]
        curr_vars = repo_1[idx]["variables"]
        while curr_len < curr_thr and idx+1 < len(repo_1):
            # iterate over ids to concate text
            idx += 1
            curr_len += repo_1[idx]["length"]
            curr_fnames += [repo_1[idx]["fname"]]
            curr_code += "\n" + repo_1[idx]["code"]
            curr_vars += repo_1[idx]["variables"]
        idx += 1
        new_repo_1["repo_name"].append(curr_repo)
        new_repo_1["fname"].append(curr_fnames)
        new_repo_1["code"].append(curr_code)
        new_repo_1["variables"].append(curr_vars)
        new_repo_1["length"].append(curr_len)
    return datasets.Dataset.from_dict(new_repo_1)

In [45]:
all_repos = {}
min_len = 0
max_len = 4096
for split in manytypes_ds.keys():
    pd_train = manytypes_ds[split].to_pandas()#["repo_name"]
    grouped_ids = pd_train.groupby("repo_name")

    filtered_repos = []
    new_repos = []

    for unique_repo_ids in tqdm(grouped_ids.groups.values()):
        repo_1 = manytypes_ds[split].select(unique_repo_ids)
        filtered_repos.append(repo_1)
    new_repo_1 = stack_one_repo(filtered_repos[0], min_len, max_len)
    vars_col_type = new_repo_1.features["variables"]
    datasets.disable_progress_bars()
    for f_repo in tqdm(filtered_repos):
        new_repo_1 = stack_one_repo(f_repo, min_len, max_len)
        new_repos.append(new_repo_1.cast_column("variables", vars_col_type))
    datasets.enable_progress_bars()
    all_repos[split] = datasets.concatenate_datasets(new_repos)


100%|██████████| 505/505 [00:11<00:00, 44.97it/s]


In [46]:
#code_repos_2k = datasets.concatenate_datasets(new_repos)

In [47]:
code_repos_4k = datasets.DatasetDict(all_repos)

In [48]:
def flatten_samples(example):
    prompt_template = lambda variable: f"What is the type of variable {variable}?"
    new_examples = {key: [] for key in example.keys()}#copy.deepcopy(example)
    new_examples["answer"] = []
    for v in example["variables"]:
        for key in new_examples.keys():
            if key == "variables":
                new_examples[key].append(prompt_template(v[0]))
                new_examples["answer"].append(v[1])
            elif key == "answer":
                pass
            else:
                new_examples[key].append(example[key])
    return new_examples

In [49]:
# filter samples with empty variables
code_repos_4k = code_repos_4k.filter(lambda example: (len(example["variables"]) > 0) and not(all([el[1] == '' for el in example["variables"]])))

Filter:   0%|          | 0/41235 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5238 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3446 [00:00<?, ? examples/s]

In [50]:
# generate prompts and answers, and recalc length
multi_code_repos_2k = code_repos_4k.map(flatten_samples)
flatten_dataset_2k = {}
for split in multi_code_repos_2k.keys():
    flat_code_repos_2k = {key: [] for key in multi_code_repos_2k[split][0].keys()}
    for el in tqdm(multi_code_repos_2k[split]):
        for key in flat_code_repos_2k.keys():
            flat_code_repos_2k[key] += el[key]
    fl_ds = datasets.Dataset.from_dict(flat_code_repos_2k)
    flatten_dataset_2k[split] = fl_ds

Map:   0%|          | 0/23932 [00:00<?, ? examples/s]

Map:   0%|          | 0/3329 [00:00<?, ? examples/s]

Map:   0%|          | 0/2318 [00:00<?, ? examples/s]

100%|██████████| 2318/2318 [00:00<00:00, 4378.78it/s]


In [51]:
# recalc len based on chat template and filter by len
def prepare_text(example):
    messages = [
        {"role": "user", "content": f"Question:\n{example['question']}\nContext:\n{example['context']}\n"},
        {"role": "assistant", "content": f"Answer:\n{example['answer']}\n"},
    ]
    ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )
    example["full_len"] = len(ids)
    return example


def tokenize_text(example):
    ids = tokenizer(example["full_text"])["input_ids"]
    example["full_len"] = len(ids)
    return example

In [52]:
flatten_dataset_2k = datasets.DatasetDict(flatten_dataset_2k)
flatten_dataset_2k = flatten_dataset_2k.rename_column("code", "context")
flatten_dataset_2k = flatten_dataset_2k.rename_column("variables", "question")

#flatten_dataset_2k = flatten_dataset_2k.map(prepare_text)

In [53]:
flatten_dataset_2k = flatten_dataset_2k.map(prepare_text, batched=False)

Map:   0%|          | 0/236974 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (3116007 > 131072). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/36201 [00:00<?, ? examples/s]

Map:   0%|          | 0/24283 [00:00<?, ? examples/s]

In [54]:
#flatten_dataset_2k_prepared = flatten_dataset_2k.map(tokenize_text)
#tokenizer(flatten_dataset_2k["train"][0]["full_text"])["input_ids"]

In [55]:
flatten_dataset_2k.save_to_disk("../../data/mt_intermediate/manytypes_flat_stacked_0k_4k")

Saving the dataset (0/6 shards):   0%|          | 0/236974 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/36201 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/24283 [00:00<?, ? examples/s]

In [56]:
flatten_dataset_2k_filtered = flatten_dataset_2k.filter(lambda example: example["full_len"] < 2*1920)

Filter:   0%|          | 0/236974 [00:00<?, ? examples/s]

Filter:   0%|          | 0/36201 [00:00<?, ? examples/s]

Filter:   0%|          | 0/24283 [00:00<?, ? examples/s]

In [57]:
flatten_dataset_2k_filtered.save_to_disk("../../data/mt_intermediate/manytypes_flat_stacked_0k_4k")

Saving the dataset (0/5 shards):   0%|          | 0/173009 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/22189 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/17091 [00:00<?, ? examples/s]

In [58]:
# now tokenize and calc length

In [62]:
flatten_dataset_2k.save_to_disk("../../data/mt_intermediate/manytypes_flat_stacked_0k_4k")

Saving the dataset (0/6 shards):   0%|          | 0/236974 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/36201 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/24283 [00:00<?, ? examples/s]

In [60]:
flatten_dataset_2k_filtered_empty = flatten_dataset_2k_filtered.filter(lambda example: example["answer"] != "")

Filter:   0%|          | 0/173009 [00:00<?, ? examples/s]

Filter:   0%|          | 0/22189 [00:00<?, ? examples/s]

Filter:   0%|          | 0/17091 [00:00<?, ? examples/s]

In [61]:
flatten_dataset_2k_filtered_empty.save_to_disk("../../data/mt_prepared/manytypes_flat_stacked_no_empty_0k_4k")

Saving the dataset (0/4 shards):   0%|          | 0/139145 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/17836 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/13505 [00:00<?, ? examples/s]

In [8]:
flatten_dataset_2k_filtered_empty = datasets.load_from_disk("../../data/mt_prepared/manytypes_flat_stacked_no_empty_0k_4k")


In [9]:
flatten_dataset_2k_filtered_empty

DatasetDict({
    train: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 139145
    })
    validation: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 17836
    })
    test: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 13505
    })
})

In [10]:
# subsample validation to 10% to speed up training
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(flatten_dataset_2k_filtered_empty["validation"])),
                           int(0.1 * len(flatten_dataset_2k_filtered_empty["validation"])),
                           replace=False)
flatten_dataset_2k_filtered_empty_subsample = copy.deepcopy(flatten_dataset_2k_filtered_empty)
flatten_dataset_2k_filtered_empty_subsample["validation"] = flatten_dataset_2k_filtered_empty["validation"].select(val_ids)

In [11]:
flatten_dataset_2k_filtered_empty_subsample.save_to_disk("../../data/mt_prepared/manytypes_flat_stacked_no_empty_sv_0k_4k")

Saving the dataset (0/3 shards):   0%|          | 0/139145 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1783 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/13505 [00:00<?, ? examples/s]

### Manytypes on 8k length

In [11]:
# now prepare prompts and filter by length
manytypes_ds = datasets.load_from_disk("../data/manytypes_raw")

In [12]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

In [13]:
def caclulate_length(example):
    messages = [
        {"role": "user", "content": example["code"]}
    ]
    length = len(tokenizer.apply_chat_template(messages, tokenize=True))
    example["length"] = length
    return example

In [14]:
manytypes_ds = manytypes_ds.map(caclulate_length)

In [15]:
def stack_one_repo(repo_1, min_len=0, max_len=2048):
    curr_thr = np.random.randint(min_len, max_len)
    # after choose samples up to curr_thr and concatenate them
    idx = 0
    new_repo_1 = {
        "repo_name": [],
        "fname": [],
        "code": [],
        "variables": [],
        "length": [],
    }
    while idx < len(repo_1):
        curr_len = repo_1[idx]["length"]
        curr_repo = repo_1[idx]["repo_name"]
        curr_fnames = [repo_1[idx]["fname"]]
        curr_code = repo_1[idx]["code"]
        curr_vars = repo_1[idx]["variables"]
        while curr_len < curr_thr and idx+1 < len(repo_1):
            # iterate over ids to concate text
            idx += 1
            curr_len += repo_1[idx]["length"]
            curr_fnames += [repo_1[idx]["fname"]]
            curr_code += "\n" + repo_1[idx]["code"]
            curr_vars += repo_1[idx]["variables"]
        idx += 1
        new_repo_1["repo_name"].append(curr_repo)
        new_repo_1["fname"].append(curr_fnames)
        new_repo_1["code"].append(curr_code)
        new_repo_1["variables"].append(curr_vars)
        new_repo_1["length"].append(curr_len)
    return datasets.Dataset.from_dict(new_repo_1)

In [16]:
all_repos = {}
min_len = 0
max_len = 8192
for split in manytypes_ds.keys():
    pd_train = manytypes_ds[split].to_pandas()#["repo_name"]
    grouped_ids = pd_train.groupby("repo_name")

    filtered_repos = []
    new_repos = []

    for unique_repo_ids in tqdm(grouped_ids.groups.values()):
        repo_1 = manytypes_ds[split].select(unique_repo_ids)
        filtered_repos.append(repo_1)
    new_repo_1 = stack_one_repo(filtered_repos[0], min_len, max_len)
    vars_col_type = new_repo_1.features["variables"]
    datasets.disable_progress_bars()
    for f_repo in tqdm(filtered_repos):
        new_repo_1 = stack_one_repo(f_repo, min_len, max_len)
        new_repos.append(new_repo_1.cast_column("variables", vars_col_type))
    datasets.enable_progress_bars()
    all_repos[split] = datasets.concatenate_datasets(new_repos)


100%|██████████| 505/505 [00:11<00:00, 44.82it/s]


In [17]:
#code_repos_2k = datasets.concatenate_datasets(new_repos)

In [18]:
code_repos_4k = datasets.DatasetDict(all_repos)

In [19]:
def flatten_samples(example):
    prompt_template = lambda variable: f"What is the type of variable {variable}?"
    new_examples = {key: [] for key in example.keys()}#copy.deepcopy(example)
    new_examples["answer"] = []
    for v in example["variables"]:
        for key in new_examples.keys():
            if key == "variables":
                new_examples[key].append(prompt_template(v[0]))
                new_examples["answer"].append(v[1])
            elif key == "answer":
                pass
            else:
                new_examples[key].append(example[key])
    return new_examples

In [20]:
# filter samples with empty variables
code_repos_4k = code_repos_4k.filter(lambda example: (len(example["variables"]) > 0) and not(all([el[1] == '' for el in example["variables"]])))

Filter:   0%|          | 0/28054 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3461 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2507 [00:00<?, ? examples/s]

In [21]:
# generate prompts and answers, and recalc length
multi_code_repos_2k = code_repos_4k.map(flatten_samples)
flatten_dataset_2k = {}
for split in multi_code_repos_2k.keys():
    flat_code_repos_2k = {key: [] for key in multi_code_repos_2k[split][0].keys()}
    for el in tqdm(multi_code_repos_2k[split]):
        for key in flat_code_repos_2k.keys():
            flat_code_repos_2k[key] += el[key]
    fl_ds = datasets.Dataset.from_dict(flat_code_repos_2k)
    flatten_dataset_2k[split] = fl_ds

Map:   0%|          | 0/17106 [00:00<?, ? examples/s]

Map:   0%|          | 0/2470 [00:00<?, ? examples/s]

Map:   0%|          | 0/1755 [00:00<?, ? examples/s]

100%|██████████| 1755/1755 [00:00<00:00, 2891.84it/s]


In [22]:
# recalc len based on chat template and filter by len
def prepare_text(example):
    messages = [
        {"role": "user", "content": f"Question:\n{example['question']}\nContext:\n{example['context']}\n"},
        {"role": "assistant", "content": f"Answer:\n{example['answer']}\n"},
    ]
    ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )
    example["full_len"] = len(ids)
    return example


def tokenize_text(example):
    ids = tokenizer(example["full_text"])["input_ids"]
    example["full_len"] = len(ids)
    return example

In [23]:
flatten_dataset_2k = datasets.DatasetDict(flatten_dataset_2k)
flatten_dataset_2k = flatten_dataset_2k.rename_column("code", "context")
flatten_dataset_2k = flatten_dataset_2k.rename_column("variables", "question")

#flatten_dataset_2k = flatten_dataset_2k.map(prepare_text)

In [24]:
flatten_dataset_2k = flatten_dataset_2k.map(prepare_text, batched=False)

Map:   0%|          | 0/240481 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (3116020 > 131072). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/36692 [00:00<?, ? examples/s]

Map:   0%|          | 0/24528 [00:00<?, ? examples/s]

In [25]:
#flatten_dataset_2k_prepared = flatten_dataset_2k.map(tokenize_text)
#tokenizer(flatten_dataset_2k["train"][0]["full_text"])["input_ids"]

In [26]:
flatten_dataset_2k.save_to_disk("../../data/mt_intermediate/manytypes_flat_stacked_0k_8k")

Saving the dataset (0/10 shards):   0%|          | 0/240481 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/36692 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/24528 [00:00<?, ? examples/s]

In [27]:
flatten_dataset_2k_filtered = flatten_dataset_2k.filter(lambda example: example["full_len"] < 4*1920)

Filter:   0%|          | 0/240481 [00:00<?, ? examples/s]

Filter:   0%|          | 0/36692 [00:00<?, ? examples/s]

Filter:   0%|          | 0/24528 [00:00<?, ? examples/s]

In [28]:
flatten_dataset_2k_filtered.save_to_disk("../../data/mt_intermediate/manytypes_flat_stacked_0k_8k")

Saving the dataset (0/9 shards):   0%|          | 0/203728 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/29187 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/20731 [00:00<?, ? examples/s]

In [29]:
# now tokenize and calc length

In [30]:
flatten_dataset_2k.save_to_disk("../../data/mt_intermediate/manytypes_flat_stacked_0k_8k")

Saving the dataset (0/10 shards):   0%|          | 0/240481 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/36692 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/24528 [00:00<?, ? examples/s]

In [31]:
flatten_dataset_2k_filtered_empty = flatten_dataset_2k_filtered.filter(lambda example: example["answer"] != "")

Filter:   0%|          | 0/203728 [00:00<?, ? examples/s]

Filter:   0%|          | 0/29187 [00:00<?, ? examples/s]

Filter:   0%|          | 0/20731 [00:00<?, ? examples/s]

In [32]:
flatten_dataset_2k_filtered_empty.save_to_disk("../../data/mt_prepared/manytypes_flat_stacked_no_empty_0k_8k")

Saving the dataset (0/7 shards):   0%|          | 0/159267 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/22681 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/16039 [00:00<?, ? examples/s]

In [33]:
flatten_dataset_2k_filtered_empty = datasets.load_from_disk("../../data/mt_prepared/manytypes_flat_stacked_no_empty_0k_8k")


In [34]:
flatten_dataset_2k_filtered_empty

DatasetDict({
    train: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 159267
    })
    validation: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 22681
    })
    test: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 16039
    })
})

In [35]:
# subsample validation to 10% to speed up training
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(flatten_dataset_2k_filtered_empty["validation"])),
                           int(0.1 * len(flatten_dataset_2k_filtered_empty["validation"])),
                           replace=False)
flatten_dataset_2k_filtered_empty_subsample = copy.deepcopy(flatten_dataset_2k_filtered_empty)
flatten_dataset_2k_filtered_empty_subsample["validation"] = flatten_dataset_2k_filtered_empty["validation"].select(val_ids)

In [36]:
flatten_dataset_2k_filtered_empty_subsample.save_to_disk("../../data/mt_prepared/manytypes_flat_stacked_no_empty_sv_0k_8k")

Saving the dataset (0/5 shards):   0%|          | 0/159267 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2268 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/16039 [00:00<?, ? examples/s]

### Manytypes on 16k length

In [37]:
# now prepare prompts and filter by length
manytypes_ds = datasets.load_from_disk("../data/manytypes_raw")

In [38]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

In [39]:
def caclulate_length(example):
    messages = [
        {"role": "user", "content": example["code"]}
    ]
    length = len(tokenizer.apply_chat_template(messages, tokenize=True))
    example["length"] = length
    return example

In [40]:
manytypes_ds = manytypes_ds.map(caclulate_length)

In [41]:
def stack_one_repo(repo_1, min_len=0, max_len=2048):
    curr_thr = np.random.randint(min_len, max_len)
    # after choose samples up to curr_thr and concatenate them
    idx = 0
    new_repo_1 = {
        "repo_name": [],
        "fname": [],
        "code": [],
        "variables": [],
        "length": [],
    }
    while idx < len(repo_1):
        curr_len = repo_1[idx]["length"]
        curr_repo = repo_1[idx]["repo_name"]
        curr_fnames = [repo_1[idx]["fname"]]
        curr_code = repo_1[idx]["code"]
        curr_vars = repo_1[idx]["variables"]
        while curr_len < curr_thr and idx+1 < len(repo_1):
            # iterate over ids to concate text
            idx += 1
            curr_len += repo_1[idx]["length"]
            curr_fnames += [repo_1[idx]["fname"]]
            curr_code += "\n" + repo_1[idx]["code"]
            curr_vars += repo_1[idx]["variables"]
        idx += 1
        new_repo_1["repo_name"].append(curr_repo)
        new_repo_1["fname"].append(curr_fnames)
        new_repo_1["code"].append(curr_code)
        new_repo_1["variables"].append(curr_vars)
        new_repo_1["length"].append(curr_len)
    return datasets.Dataset.from_dict(new_repo_1)

In [42]:
all_repos = {}
min_len = 0
max_len = 16384
for split in manytypes_ds.keys():
    pd_train = manytypes_ds[split].to_pandas()#["repo_name"]
    grouped_ids = pd_train.groupby("repo_name")

    filtered_repos = []
    new_repos = []

    for unique_repo_ids in tqdm(grouped_ids.groups.values()):
        repo_1 = manytypes_ds[split].select(unique_repo_ids)
        filtered_repos.append(repo_1)
    new_repo_1 = stack_one_repo(filtered_repos[0], min_len, max_len)
    vars_col_type = new_repo_1.features["variables"]
    datasets.disable_progress_bars()
    for f_repo in tqdm(filtered_repos):
        new_repo_1 = stack_one_repo(f_repo, min_len, max_len)
        new_repos.append(new_repo_1.cast_column("variables", vars_col_type))
    datasets.enable_progress_bars()
    all_repos[split] = datasets.concatenate_datasets(new_repos)


100%|██████████| 505/505 [00:11<00:00, 44.42it/s]


In [43]:
#code_repos_2k = datasets.concatenate_datasets(new_repos)

In [44]:
code_repos_4k = datasets.DatasetDict(all_repos)

In [45]:
def flatten_samples(example):
    prompt_template = lambda variable: f"What is the type of variable {variable}?"
    new_examples = {key: [] for key in example.keys()}#copy.deepcopy(example)
    new_examples["answer"] = []
    for v in example["variables"]:
        for key in new_examples.keys():
            if key == "variables":
                new_examples[key].append(prompt_template(v[0]))
                new_examples["answer"].append(v[1])
            elif key == "answer":
                pass
            else:
                new_examples[key].append(example[key])
    return new_examples

In [46]:
# filter samples with empty variables
code_repos_4k = code_repos_4k.filter(lambda example: (len(example["variables"]) > 0) and not(all([el[1] == '' for el in example["variables"]])))

Filter:   0%|          | 0/19331 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2359 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1698 [00:00<?, ? examples/s]

In [47]:
# generate prompts and answers, and recalc length
multi_code_repos_2k = code_repos_4k.map(flatten_samples)
flatten_dataset_2k = {}
for split in multi_code_repos_2k.keys():
    flat_code_repos_2k = {key: [] for key in multi_code_repos_2k[split][0].keys()}
    for el in tqdm(multi_code_repos_2k[split]):
        for key in flat_code_repos_2k.keys():
            flat_code_repos_2k[key] += el[key]
    fl_ds = datasets.Dataset.from_dict(flat_code_repos_2k)
    flatten_dataset_2k[split] = fl_ds

Map:   0%|          | 0/11940 [00:00<?, ? examples/s]

Map:   0%|          | 0/1644 [00:00<?, ? examples/s]

Map:   0%|          | 0/1318 [00:00<?, ? examples/s]

100%|██████████| 1318/1318 [00:01<00:00, 1255.24it/s]


In [48]:
# recalc len based on chat template and filter by len
def prepare_text(example):
    messages = [
        {"role": "user", "content": f"Question:\n{example['question']}\nContext:\n{example['context']}\n"},
        {"role": "assistant", "content": f"Answer:\n{example['answer']}\n"},
    ]
    ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )
    example["full_len"] = len(ids)
    return example


def tokenize_text(example):
    ids = tokenizer(example["full_text"])["input_ids"]
    example["full_len"] = len(ids)
    return example

In [49]:
flatten_dataset_2k = datasets.DatasetDict(flatten_dataset_2k)
flatten_dataset_2k = flatten_dataset_2k.rename_column("code", "context")
flatten_dataset_2k = flatten_dataset_2k.rename_column("variables", "question")

#flatten_dataset_2k = flatten_dataset_2k.map(prepare_text)

In [50]:
flatten_dataset_2k = flatten_dataset_2k.map(prepare_text, batched=False)

Map:   0%|          | 0/243531 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (3122868 > 131072). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/37005 [00:00<?, ? examples/s]

Map:   0%|          | 0/24986 [00:00<?, ? examples/s]

In [51]:
#flatten_dataset_2k_prepared = flatten_dataset_2k.map(tokenize_text)
#tokenizer(flatten_dataset_2k["train"][0]["full_text"])["input_ids"]

In [52]:
flatten_dataset_2k.save_to_disk("../../data/mt_intermediate/manytypes_flat_stacked_0k_16k")

Saving the dataset (0/17 shards):   0%|          | 0/243531 [00:00<?, ? examples/s]

Saving the dataset (0/3 shards):   0%|          | 0/37005 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/24986 [00:00<?, ? examples/s]

In [53]:
flatten_dataset_2k_filtered = flatten_dataset_2k.filter(lambda example: example["full_len"] < 8*1920)

Filter:   0%|          | 0/243531 [00:00<?, ? examples/s]

Filter:   0%|          | 0/37005 [00:00<?, ? examples/s]

Filter:   0%|          | 0/24986 [00:00<?, ? examples/s]

In [54]:
flatten_dataset_2k_filtered.save_to_disk("../../data/mt_intermediate/manytypes_flat_stacked_0k_16k")

Saving the dataset (0/15 shards):   0%|          | 0/215388 [00:00<?, ? examples/s]

Saving the dataset (0/3 shards):   0%|          | 0/28893 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/23051 [00:00<?, ? examples/s]

In [55]:
# now tokenize and calc length

In [56]:
flatten_dataset_2k.save_to_disk("../../data/mt_intermediate/manytypes_flat_stacked_0k_16k")

Saving the dataset (0/17 shards):   0%|          | 0/243531 [00:00<?, ? examples/s]

Saving the dataset (0/3 shards):   0%|          | 0/37005 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/24986 [00:00<?, ? examples/s]

In [57]:
flatten_dataset_2k_filtered_empty = flatten_dataset_2k_filtered.filter(lambda example: example["answer"] != "")

Filter:   0%|          | 0/215388 [00:00<?, ? examples/s]

Filter:   0%|          | 0/28893 [00:00<?, ? examples/s]

Filter:   0%|          | 0/23051 [00:00<?, ? examples/s]

In [58]:
flatten_dataset_2k_filtered_empty.save_to_disk("../../data/mt_prepared/manytypes_flat_stacked_no_empty_0k_16k")

Saving the dataset (0/12 shards):   0%|          | 0/165684 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/22406 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/17194 [00:00<?, ? examples/s]

In [59]:
flatten_dataset_2k_filtered_empty = datasets.load_from_disk("../../data/mt_prepared/manytypes_flat_stacked_no_empty_0k_16k")


In [60]:
flatten_dataset_2k_filtered_empty

DatasetDict({
    train: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 165684
    })
    validation: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 22406
    })
    test: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 17194
    })
})

In [61]:
# subsample validation to 10% to speed up training
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(flatten_dataset_2k_filtered_empty["validation"])),
                           int(0.1 * len(flatten_dataset_2k_filtered_empty["validation"])),
                           replace=False)
flatten_dataset_2k_filtered_empty_subsample = copy.deepcopy(flatten_dataset_2k_filtered_empty)
flatten_dataset_2k_filtered_empty_subsample["validation"] = flatten_dataset_2k_filtered_empty["validation"].select(val_ids)

In [62]:
flatten_dataset_2k_filtered_empty_subsample.save_to_disk("../../data/mt_prepared/manytypes_flat_stacked_no_empty_sv_0k_16k")

Saving the dataset (0/9 shards):   0%|          | 0/165684 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2240 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/17194 [00:00<?, ? examples/s]

### Manytypes on 32k length

In [4]:
# now prepare prompts and filter by length
manytypes_ds = datasets.load_from_disk("../data/manytypes_raw")

In [5]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

In [6]:
def caclulate_length(example):
    messages = [
        {"role": "user", "content": example["code"]}
    ]
    length = len(tokenizer.apply_chat_template(messages, tokenize=True))
    example["length"] = length
    return example

In [7]:
manytypes_ds = manytypes_ds.map(caclulate_length)

In [8]:
def stack_one_repo(repo_1, min_len=0, max_len=2048):
    curr_thr = np.random.randint(min_len, max_len)
    # after choose samples up to curr_thr and concatenate them
    idx = 0
    new_repo_1 = {
        "repo_name": [],
        "fname": [],
        "code": [],
        "variables": [],
        "length": [],
    }
    while idx < len(repo_1):
        curr_len = repo_1[idx]["length"]
        curr_repo = repo_1[idx]["repo_name"]
        curr_fnames = [repo_1[idx]["fname"]]
        curr_code = repo_1[idx]["code"]
        curr_vars = repo_1[idx]["variables"]
        while curr_len < curr_thr and idx+1 < len(repo_1):
            # iterate over ids to concate text
            idx += 1
            curr_len += repo_1[idx]["length"]
            curr_fnames += [repo_1[idx]["fname"]]
            curr_code += "\n" + repo_1[idx]["code"]
            curr_vars += repo_1[idx]["variables"]
        idx += 1
        new_repo_1["repo_name"].append(curr_repo)
        new_repo_1["fname"].append(curr_fnames)
        new_repo_1["code"].append(curr_code)
        new_repo_1["variables"].append(curr_vars)
        new_repo_1["length"].append(curr_len)
    return datasets.Dataset.from_dict(new_repo_1)

In [9]:
all_repos = {}
min_len = 0
max_len = 2 * 16384
for split in manytypes_ds.keys():
    pd_train = manytypes_ds[split].to_pandas()#["repo_name"]
    grouped_ids = pd_train.groupby("repo_name")

    filtered_repos = []
    new_repos = []

    for unique_repo_ids in tqdm(grouped_ids.groups.values()):
        repo_1 = manytypes_ds[split].select(unique_repo_ids)
        filtered_repos.append(repo_1)
    new_repo_1 = stack_one_repo(filtered_repos[0], min_len, max_len)
    vars_col_type = new_repo_1.features["variables"]
    datasets.disable_progress_bars()
    for f_repo in tqdm(filtered_repos):
        new_repo_1 = stack_one_repo(f_repo, min_len, max_len)
        new_repos.append(new_repo_1.cast_column("variables", vars_col_type))
    datasets.enable_progress_bars()
    all_repos[split] = datasets.concatenate_datasets(new_repos)


100%|██████████| 505/505 [00:11<00:00, 44.47it/s]


In [10]:
#code_repos_2k = datasets.concatenate_datasets(new_repos)

In [11]:
code_repos_4k = datasets.DatasetDict(all_repos)

In [12]:
def flatten_samples(example):
    prompt_template = lambda variable: f"What is the type of variable {variable}?"
    new_examples = {key: [] for key in example.keys()}#copy.deepcopy(example)
    new_examples["answer"] = []
    for v in example["variables"]:
        for key in new_examples.keys():
            if key == "variables":
                new_examples[key].append(prompt_template(v[0]))
                new_examples["answer"].append(v[1])
            elif key == "answer":
                pass
            else:
                new_examples[key].append(example[key])
    return new_examples

In [13]:
# filter samples with empty variables
code_repos_4k = code_repos_4k.filter(lambda example: (len(example["variables"]) > 0) and not(all([el[1] == '' for el in example["variables"]])))

Filter:   0%|          | 0/13447 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1429 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1082 [00:00<?, ? examples/s]

In [17]:
code_repos_4k

DatasetDict({
    train: Dataset({
        features: ['repo_name', 'fname', 'code', 'variables', 'length'],
        num_rows: 8947
    })
    validation: Dataset({
        features: ['repo_name', 'fname', 'code', 'variables', 'length'],
        num_rows: 1088
    })
    test: Dataset({
        features: ['repo_name', 'fname', 'code', 'variables', 'length'],
        num_rows: 887
    })
})

In [18]:
# generate prompts and answers, and recalc length
multi_code_repos_2k = datasets.DatasetDict({"test": code_repos_4k["test"].map(flatten_samples)})
flatten_dataset_2k = {}
for split in multi_code_repos_2k.keys():
    flat_code_repos_2k = {key: [] for key in multi_code_repos_2k[split][0].keys()}
    for el in tqdm(multi_code_repos_2k[split]):
        for key in flat_code_repos_2k.keys():
            flat_code_repos_2k[key] += el[key]
    fl_ds = datasets.Dataset.from_dict(flat_code_repos_2k)
    flatten_dataset_2k[split] = fl_ds

Map:   0%|          | 0/887 [00:00<?, ? examples/s]

100%|██████████| 887/887 [00:01<00:00, 563.32it/s]


In [19]:
# recalc len based on chat template and filter by len
def prepare_text(example):
    messages = [
        {"role": "user", "content": f"Question:\n{example['question']}\nContext:\n{example['context']}\n"},
        {"role": "assistant", "content": f"Answer:\n{example['answer']}\n"},
    ]
    ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )
    example["full_len"] = len(ids)
    return example


def tokenize_text(example):
    ids = tokenizer(example["full_text"])["input_ids"]
    example["full_len"] = len(ids)
    return example

In [20]:
flatten_dataset_2k = datasets.DatasetDict(flatten_dataset_2k)
flatten_dataset_2k = flatten_dataset_2k.rename_column("code", "context")
flatten_dataset_2k = flatten_dataset_2k.rename_column("variables", "question")

#flatten_dataset_2k = flatten_dataset_2k.map(prepare_text)

In [21]:
flatten_dataset_2k = flatten_dataset_2k.map(prepare_text, batched=False)

Map:   0%|          | 0/25379 [00:00<?, ? examples/s]

In [ ]:
#flatten_dataset_2k_prepared = flatten_dataset_2k.map(tokenize_text)
#tokenizer(flatten_dataset_2k["train"][0]["full_text"])["input_ids"]

In [22]:
flatten_dataset_2k.save_to_disk("../../data/mt_intermediate/test_only_manytypes_flat_stacked_0k_32k")

Saving the dataset (0/3 shards):   0%|          | 0/25379 [00:00<?, ? examples/s]

In [23]:
flatten_dataset_2k_filtered = flatten_dataset_2k.filter(lambda example: example["full_len"] < 16*1920)

Filter:   0%|          | 0/25379 [00:00<?, ? examples/s]

In [24]:
flatten_dataset_2k_filtered.save_to_disk("../../data/mt_intermediate/test_only_manytypes_flat_stacked_0k_32k")

Saving the dataset (0/3 shards):   0%|          | 0/24241 [00:00<?, ? examples/s]

In [25]:
# now tokenize and calc length

In [26]:
flatten_dataset_2k.save_to_disk("../../data/mt_intermediate/test_only_manytypes_flat_stacked_0k_32k")

Saving the dataset (0/3 shards):   0%|          | 0/25379 [00:00<?, ? examples/s]

In [27]:
flatten_dataset_2k_filtered_empty = flatten_dataset_2k_filtered.filter(lambda example: example["answer"] != "")

Filter:   0%|          | 0/24241 [00:00<?, ? examples/s]

In [28]:
flatten_dataset_2k_filtered_empty.save_to_disk("../../data/mt_prepared/test_only_manytypes_flat_stacked_no_empty_0k_32k")

Saving the dataset (0/2 shards):   0%|          | 0/17826 [00:00<?, ? examples/s]

In [29]:
flatten_dataset_2k_filtered_empty = datasets.load_from_disk("../../data/mt_prepared/test_only_manytypes_flat_stacked_no_empty_0k_32k")


In [30]:
flatten_dataset_2k_filtered_empty

DatasetDict({
    test: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 17826
    })
})

In [ ]:
# subsample validation to 10% to speed up training
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(flatten_dataset_2k_filtered_empty["validation"])),
                           int(0.1 * len(flatten_dataset_2k_filtered_empty["validation"])),
                           replace=False)
flatten_dataset_2k_filtered_empty_subsample = copy.deepcopy(flatten_dataset_2k_filtered_empty)
flatten_dataset_2k_filtered_empty_subsample["validation"] = flatten_dataset_2k_filtered_empty["validation"].select(val_ids)

In [ ]:
flatten_dataset_2k_filtered_empty_subsample.save_to_disk("../../data/mt_prepared/manytypes_flat_stacked_no_empty_sv_0k_32k")

Saving the dataset (0/9 shards):   0%|          | 0/165684 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2240 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/17194 [00:00<?, ? examples/s]

### Manytypes on 64k length, only test, 100 per split

In [17]:
# now prepare prompts and filter by length
manytypes_ds = datasets.load_from_disk("../data/manytypes_raw")

In [18]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

In [19]:
def caclulate_length(example):
    messages = [
        {"role": "user", "content": example["code"]}
    ]
    length = len(tokenizer.apply_chat_template(messages, tokenize=True))
    example["length"] = length
    return example

In [20]:
manytypes_ds = manytypes_ds.map(caclulate_length)

In [21]:
def stack_one_repo(repo_1, min_len=0, max_len=2048):
    curr_thr = np.random.randint(min_len, max_len)
    # after choose samples up to curr_thr and concatenate them
    idx = 0
    new_repo_1 = {
        "repo_name": [],
        "fname": [],
        "code": [],
        "variables": [],
        "length": [],
    }
    while idx < len(repo_1):
        curr_len = repo_1[idx]["length"]
        curr_repo = repo_1[idx]["repo_name"]
        curr_fnames = [repo_1[idx]["fname"]]
        curr_code = repo_1[idx]["code"]
        curr_vars = repo_1[idx]["variables"]
        while curr_len < curr_thr and idx+1 < len(repo_1):
            # iterate over ids to concate text
            idx += 1
            curr_len += repo_1[idx]["length"]
            curr_fnames += [repo_1[idx]["fname"]]
            curr_code += "\n" + repo_1[idx]["code"]
            curr_vars += repo_1[idx]["variables"]
        idx += 1
        new_repo_1["repo_name"].append(curr_repo)
        new_repo_1["fname"].append(curr_fnames)
        new_repo_1["code"].append(curr_code)
        new_repo_1["variables"].append(curr_vars)
        new_repo_1["length"].append(curr_len)
    return datasets.Dataset.from_dict(new_repo_1)

In [22]:
all_repos = {}
min_len = 0
max_len = 64 * 1024
split_suffix = "64k"
for split in manytypes_ds.keys():
    pd_train = manytypes_ds[split].to_pandas()#["repo_name"]
    grouped_ids = pd_train.groupby("repo_name")

    filtered_repos = []
    new_repos = []

    for unique_repo_ids in tqdm(grouped_ids.groups.values()):
        repo_1 = manytypes_ds[split].select(unique_repo_ids)
        filtered_repos.append(repo_1)
    new_repo_1 = stack_one_repo(filtered_repos[0], min_len, max_len)
    vars_col_type = new_repo_1.features["variables"]
    datasets.disable_progress_bars()
    for f_repo in tqdm(filtered_repos):
        new_repo_1 = stack_one_repo(f_repo, min_len, max_len)
        new_repos.append(new_repo_1.cast_column("variables", vars_col_type))
    datasets.enable_progress_bars()
    all_repos[split] = datasets.concatenate_datasets(new_repos)


100%|██████████| 505/505 [00:11<00:00, 44.43it/s]


In [23]:
#code_repos_2k = datasets.concatenate_datasets(new_repos)

In [24]:
code_repos_4k = datasets.DatasetDict(all_repos)

In [25]:
def flatten_samples(example):
    prompt_template = lambda variable: f"What is the type of variable {variable}?"
    new_examples = {key: [] for key in example.keys()}#copy.deepcopy(example)
    new_examples["answer"] = []
    for v in example["variables"]:
        for key in new_examples.keys():
            if key == "variables":
                new_examples[key].append(prompt_template(v[0]))
                new_examples["answer"].append(v[1])
            elif key == "answer":
                pass
            else:
                new_examples[key].append(example[key])
    return new_examples

In [26]:
# filter samples with empty variables
code_repos_4k = code_repos_4k.filter(lambda example: (len(example["variables"]) > 0) and not(all([el[1] == '' for el in example["variables"]])))

Filter:   0%|          | 0/9753 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1202 [00:00<?, ? examples/s]

Filter:   0%|          | 0/878 [00:00<?, ? examples/s]

In [27]:
# generate prompts and answers, and recalc length
#multi_code_repos_2k = code_repos_4k.map(flatten_samples)
multi_code_repos_2k = datasets.DatasetDict({"test": code_repos_4k["test"].map(flatten_samples)})

flatten_dataset_2k = {}
for split in multi_code_repos_2k.keys():
    flat_code_repos_2k = {key: [] for key in multi_code_repos_2k[split][0].keys()}
    for el in tqdm(multi_code_repos_2k[split]):
        for key in flat_code_repos_2k.keys():
            flat_code_repos_2k[key] += el[key]
    fl_ds = datasets.Dataset.from_dict(flat_code_repos_2k)
    flatten_dataset_2k[split] = fl_ds

Map:   0%|          | 0/723 [00:00<?, ? examples/s]

100%|██████████| 723/723 [00:02<00:00, 290.33it/s]


In [28]:
# recalc len based on chat template and filter by len
def prepare_text(example):
    messages = [
        {"role": "user", "content": f"Question:\n{example['question']}\nContext:\n{example['context']}\n"},
        {"role": "assistant", "content": f"Answer:\n{example['answer']}\n"},
    ]
    ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )
    example["full_len"] = len(ids)
    return example


def tokenize_text(example):
    ids = tokenizer(example["full_text"])["input_ids"]
    example["full_len"] = len(ids)
    return example

In [29]:
flatten_dataset_2k = datasets.DatasetDict(flatten_dataset_2k)
flatten_dataset_2k = flatten_dataset_2k.rename_column("code", "context")
flatten_dataset_2k = flatten_dataset_2k.rename_column("variables", "question")

#flatten_dataset_2k = flatten_dataset_2k.map(prepare_text)

In [30]:
flatten_dataset_2k = flatten_dataset_2k.map(prepare_text, batched=False)

Map:   0%|          | 0/25381 [00:00<?, ? examples/s]

In [31]:
#flatten_dataset_2k_prepared = flatten_dataset_2k.map(tokenize_text)
#tokenizer(flatten_dataset_2k["train"][0]["full_text"])["input_ids"]

In [32]:
flatten_dataset_2k.save_to_disk(f"../../data/mt_intermediate/manytypes_flat_stacked_0k_{split_suffix}")

Saving the dataset (0/5 shards):   0%|          | 0/25381 [00:00<?, ? examples/s]

In [33]:
flatten_dataset_2k_filtered = flatten_dataset_2k.filter(lambda example: example["full_len"] < max_len-int(max_len/1024*64))

Filter:   0%|          | 0/25381 [00:00<?, ? examples/s]

In [34]:
flatten_dataset_2k_filtered.save_to_disk(f"../../data/mt_intermediate/manytypes_flat_stacked_0k_{split_suffix}")

Saving the dataset (0/5 shards):   0%|          | 0/25329 [00:00<?, ? examples/s]

In [35]:
# now tokenize and calc length

In [36]:
flatten_dataset_2k.save_to_disk(f"../../data/mt_intermediate/manytypes_flat_stacked_0k_{split_suffix}")

Saving the dataset (0/5 shards):   0%|          | 0/25381 [00:00<?, ? examples/s]

In [37]:
flatten_dataset_2k_filtered_empty = flatten_dataset_2k_filtered.filter(lambda example: example["answer"] != "")

Filter:   0%|          | 0/25329 [00:00<?, ? examples/s]

In [38]:
flatten_dataset_2k_filtered_empty.save_to_disk(f"../../data/mt_prepared/manytypes_flat_stacked_no_empty_0k_{split_suffix}")

Saving the dataset (0/3 shards):   0%|          | 0/18615 [00:00<?, ? examples/s]

In [39]:
flatten_dataset_2k_filtered_empty = datasets.load_from_disk(f"../../data/mt_prepared/manytypes_flat_stacked_no_empty_0k_{split_suffix}")


In [40]:
flatten_dataset_2k_filtered_empty

DatasetDict({
    test: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 18615
    })
})

In [41]:
# subsample validation to 10% to speed up training
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(flatten_dataset_2k_filtered_empty["validation"])),
                           int(0.1 * len(flatten_dataset_2k_filtered_empty["validation"])),
                           replace=False)
flatten_dataset_2k_filtered_empty_subsample = copy.deepcopy(flatten_dataset_2k_filtered_empty)
flatten_dataset_2k_filtered_empty_subsample["validation"] = flatten_dataset_2k_filtered_empty["validation"].select(val_ids)

KeyError: 'validation'

In [ ]:
flatten_dataset_2k_filtered_empty_subsample.save_to_disk(f"../../data/mt_prepared/manytypes_flat_stacked_no_empty_sv_0k_{split_suffix}")

Saving the dataset (0/1 shards):   0%|          | 0/35224 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/470 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4269 [00:00<?, ? examples/s]

### Manytypes on real 128k length, only test, 100 per split

In [4]:
# now prepare prompts and filter by length
manytypes_ds = datasets.load_from_disk("../data/manytypes_raw")

In [5]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

In [6]:
def caclulate_length(example):
    messages = [
        {"role": "user", "content": example["code"]}
    ]
    length = len(tokenizer.apply_chat_template(messages, tokenize=True))
    example["length"] = length
    return example

In [7]:
manytypes_ds = manytypes_ds.map(caclulate_length)

In [8]:
def stack_one_repo(repo_1, min_len=0, max_len=2048):
    curr_thr = np.random.randint(min_len, max_len)
    # after choose samples up to curr_thr and concatenate them
    idx = 0
    new_repo_1 = {
        "repo_name": [],
        "fname": [],
        "code": [],
        "variables": [],
        "length": [],
    }
    while idx < len(repo_1):
        curr_len = repo_1[idx]["length"]
        curr_repo = repo_1[idx]["repo_name"]
        curr_fnames = [repo_1[idx]["fname"]]
        curr_code = repo_1[idx]["code"]
        curr_vars = repo_1[idx]["variables"]
        while curr_len < curr_thr and idx+1 < len(repo_1):
            # iterate over ids to concate text
            idx += 1
            curr_len += repo_1[idx]["length"]
            curr_fnames += [repo_1[idx]["fname"]]
            curr_code += "\n" + repo_1[idx]["code"]
            curr_vars += repo_1[idx]["variables"]
        idx += 1
        new_repo_1["repo_name"].append(curr_repo)
        new_repo_1["fname"].append(curr_fnames)
        new_repo_1["code"].append(curr_code)
        new_repo_1["variables"].append(curr_vars)
        new_repo_1["length"].append(curr_len)
    return datasets.Dataset.from_dict(new_repo_1)

In [9]:
all_repos = {}
min_len = 64 * 1024
max_len = 128 * 1024
split_suffix = "128k"
for split in manytypes_ds.keys():
    pd_train = manytypes_ds[split].to_pandas()#["repo_name"]
    grouped_ids = pd_train.groupby("repo_name")

    filtered_repos = []
    new_repos = []

    for unique_repo_ids in tqdm(grouped_ids.groups.values()):
        repo_1 = manytypes_ds[split].select(unique_repo_ids)
        filtered_repos.append(repo_1)
    new_repo_1 = stack_one_repo(filtered_repos[0], min_len, max_len)
    vars_col_type = new_repo_1.features["variables"]
    datasets.disable_progress_bars()
    for f_repo in tqdm(filtered_repos):
        new_repo_1 = stack_one_repo(f_repo, min_len, max_len)
        new_repos.append(new_repo_1.cast_column("variables", vars_col_type))
    datasets.enable_progress_bars()
    all_repos[split] = datasets.concatenate_datasets(new_repos)


100%|██████████| 505/505 [00:06<00:00, 83.40it/s] 


In [10]:
#code_repos_2k = datasets.concatenate_datasets(new_repos)

In [11]:
code_repos_4k = datasets.DatasetDict(all_repos)

In [12]:
def flatten_samples(example):
    prompt_template = lambda variable: f"What is the type of variable {variable}?"
    new_examples = {key: [] for key in example.keys()}#copy.deepcopy(example)
    new_examples["answer"] = []
    for v in example["variables"]:
        for key in new_examples.keys():
            if key == "variables":
                new_examples[key].append(prompt_template(v[0]))
                new_examples["answer"].append(v[1])
            elif key == "answer":
                pass
            else:
                new_examples[key].append(example[key])
    return new_examples

In [13]:
# filter samples with empty variables
code_repos_4k = code_repos_4k.filter(lambda example: (len(example["variables"]) > 0) and not(all([el[1] == '' for el in example["variables"]])))

Filter:   0%|          | 0/5000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/641 [00:00<?, ? examples/s]

Filter:   0%|          | 0/547 [00:00<?, ? examples/s]

In [21]:
import pyarrow as pa
from datasets import Value, Sequence

In [20]:
buf_dataset.features

{'repo_name': Value(dtype='string', id=None),
 'fname': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
 'code': Value(dtype='string', id=None),
 'variables': Sequence(feature=Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), length=-1, id=None),
 'length': Value(dtype='int64', id=None)}

In [53]:
buf_dataset = code_repos_4k["test"]
buf_dataset = buf_dataset.cast_column("variables", Sequence(Sequence(Value("large_string"))))
buf_dataset = buf_dataset.cast_column("code", Value("large_string"))

#buf_dataset = buf_dataset.cast(
#    buf_dataset.remove_columns(["variables"]).add_column("variables", pa.large_string())
#)

Casting the dataset:   0%|          | 0/503 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/503 [00:00<?, ? examples/s]

In [54]:
#buf_dataset = buf_dataset.filter(lambda x: x["length"] < 1024 * 96)

In [55]:
np.percentile(buf_dataset["length"], 99)

np.float64(121122.96000000002)

In [56]:
np.sort(buf_dataset["length"])

array([    97,    124,    129,    132,    144,    151,    158,    179,
          222,    238,    243,    250,    251,    269,    292,    308,
          335,    347,    353,    363,    367,    371,    372,    376,
          376,    376,    410,    410,    410,    411,    459,    465,
          468,    486,    507,    520,    526,    526,    550,    555,
          560,    560,    568,    585,    585,    591,    601,    647,
          655,    657,    663,    668,    674,    678,    683,    684,
          693,    724,    724,    733,    760,    762,    772,    784,
          788,    788,    792,    796,    806,    806,    808,    824,
          825,    904,    917,    920,    989,   1027,   1030,   1033,
         1085,   1164,   1166,   1174,   1178,   1196,   1210,   1212,
         1215,   1216,   1225,   1253,   1256,   1262,   1262,   1291,
         1330,   1335,   1345,   1353,   1357,   1361,   1371,   1372,
         1374,   1382,   1392,   1448,   1453,   1493,   1527,   1546,
      

In [57]:
# Break the dataset into 10 smaller parts and process them
shards = [buf_dataset.shard(num_shards=10, index=i) for i in range(10)]

processed_shards = [
    shard.map(flatten_samples, batched=False) 
    for shard in shards
]


Map:   0%|          | 0/51 [00:00<?, ? examples/s]

Map:   0%|          | 0/51 [00:00<?, ? examples/s]

Map:   0%|          | 0/51 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [58]:
# generate prompts and answers, and recalc length
#multi_code_repos_2k = code_repos_4k.map(flatten_samples)
#multi_code_repos_2k = datasets.DatasetDict({"test": code_repos_4k["test"].map(flatten_samples)})

#mapped_ds = buf_dataset.map(flatten_samples, batched=False)
#multi_code_repos_2k = datasets.DatasetDict({"test": mapped_ds})
multi_code_repos_2k = datasets.DatasetDict({"test": datasets.concatenate_datasets(processed_shards)})

flatten_dataset_2k = {}
for split in multi_code_repos_2k.keys():
    flat_code_repos_2k = {key: [] for key in multi_code_repos_2k[split][0].keys()}
    for el in tqdm(multi_code_repos_2k[split]):
        for key in flat_code_repos_2k.keys():
            flat_code_repos_2k[key] += el[key]
    fl_ds = datasets.Dataset.from_dict(flat_code_repos_2k)
    flatten_dataset_2k[split] = fl_ds

100%|██████████| 503/503 [00:04<00:00, 118.16it/s]


In [59]:
# recalc len based on chat template and filter by len
def prepare_text(example):
    messages = [
        {"role": "user", "content": f"Question:\n{example['question']}\nContext:\n{example['context']}\n"},
        {"role": "assistant", "content": f"Answer:\n{example['answer']}\n"},
    ]
    ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )
    example["full_len"] = len(ids)
    return example


def tokenize_text(example):
    ids = tokenizer(example["full_text"])["input_ids"]
    example["full_len"] = len(ids)
    return example

In [60]:
flatten_dataset_2k = datasets.DatasetDict(flatten_dataset_2k)
flatten_dataset_2k = flatten_dataset_2k.rename_column("code", "context")
flatten_dataset_2k = flatten_dataset_2k.rename_column("variables", "question")

#flatten_dataset_2k = flatten_dataset_2k.map(prepare_text)

In [61]:
flatten_dataset_2k = flatten_dataset_2k.map(prepare_text, batched=False)

Map:   0%|          | 0/25716 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (162146 > 131072). Running this sequence through the model will result in indexing errors


In [62]:
split_suffix

'128k'

In [31]:
#flatten_dataset_2k_prepared = flatten_dataset_2k.map(tokenize_text)
#tokenizer(flatten_dataset_2k["train"][0]["full_text"])["input_ids"]

In [63]:
flatten_dataset_2k.save_to_disk(f"../../data/mt_intermediate/manytypes_flat_stacked_64k_{split_suffix}")

Saving the dataset (0/9 shards):   0%|          | 0/25716 [00:00<?, ? examples/s]

In [64]:
flatten_dataset_2k_filtered = flatten_dataset_2k.filter(lambda example: example["full_len"] < max_len)

Filter:   0%|          | 0/25716 [00:00<?, ? examples/s]

In [65]:
flatten_dataset_2k_filtered.save_to_disk(f"../../data/mt_intermediate/manytypes_flat_stacked_64k_{split_suffix}")

Saving the dataset (0/9 shards):   0%|          | 0/25513 [00:00<?, ? examples/s]

In [66]:
# now tokenize and calc length

In [67]:
flatten_dataset_2k.save_to_disk(f"../../data/mt_intermediate/manytypes_flat_stacked_64k_{split_suffix}")

Saving the dataset (0/9 shards):   0%|          | 0/25716 [00:00<?, ? examples/s]

In [68]:
flatten_dataset_2k_filtered_empty = flatten_dataset_2k_filtered.filter(lambda example: example["answer"] != "")

Filter:   0%|          | 0/25513 [00:00<?, ? examples/s]

In [69]:
flatten_dataset_2k_filtered_empty.save_to_disk(f"../../data/mt_prepared/manytypes_flat_stacked_no_empty_64k_{split_suffix}")

Saving the dataset (0/7 shards):   0%|          | 0/18512 [00:00<?, ? examples/s]

In [70]:
flatten_dataset_2k_filtered_empty = datasets.load_from_disk(f"../../data/mt_prepared/manytypes_flat_stacked_no_empty_64k_{split_suffix}")


In [71]:
flatten_dataset_2k_filtered_empty

DatasetDict({
    test: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 18512
    })
})

In [41]:
# subsample validation to 10% to speed up training
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(flatten_dataset_2k_filtered_empty["validation"])),
                           int(0.1 * len(flatten_dataset_2k_filtered_empty["validation"])),
                           replace=False)
flatten_dataset_2k_filtered_empty_subsample = copy.deepcopy(flatten_dataset_2k_filtered_empty)
flatten_dataset_2k_filtered_empty_subsample["validation"] = flatten_dataset_2k_filtered_empty["validation"].select(val_ids)

KeyError: 'validation'

In [ ]:
flatten_dataset_2k_filtered_empty_subsample.save_to_disk(f"../../data/mt_prepared/manytypes_flat_stacked_no_empty_sv_64k_{split_suffix}")

Saving the dataset (0/1 shards):   0%|          | 0/35224 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/470 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4269 [00:00<?, ? examples/s]

### Select small test subset for eval

In [12]:
flatten_dataset_2k_filtered_empty = datasets.load_from_disk("../../data/mt_prepared/manytypes_flat_stacked_no_empty_sv_0k_2k")
np.random.seed(43)
test_ids = np.random.choice(np.arange(len(flatten_dataset_2k_filtered_empty["test"])), 100, replace=False)
test_2k_mt = flatten_dataset_2k_filtered_empty["test"].select(test_ids)
test_2k_mt.save_to_disk("../../data/mt_prepared/test_mt_flat_stacked_no_empty_sv_0k_2k")

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

In [13]:
flatten_dataset_4k_filtered_empty = datasets.load_from_disk("../../data/mt_prepared/manytypes_flat_stacked_no_empty_sv_0k_4k")
np.random.seed(43)
test_ids = np.random.choice(np.arange(len(flatten_dataset_4k_filtered_empty["test"])), 100, replace=False)
test_4k_mt = flatten_dataset_4k_filtered_empty["test"].select(test_ids)
test_4k_mt.save_to_disk("../../data/mt_prepared/test_mt_flat_stacked_no_empty_sv_0k_4k")

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

In [4]:
flatten_dataset_4k_filtered_empty = datasets.load_from_disk("../../data/mt_prepared/manytypes_flat_stacked_no_empty_sv_0k_4k")
np.random.seed(43)
test_ids = np.random.choice(np.arange(len(flatten_dataset_4k_filtered_empty["test"])), 400, replace=False)
test_4k_mt = flatten_dataset_4k_filtered_empty["test"].select(test_ids)
test_4k_mt.save_to_disk("../../data/mt_prepared/test_mt_flat_stacked_no_empty_sv_0k_4k_400")

Saving the dataset (0/1 shards):   0%|          | 0/400 [00:00<?, ? examples/s]

In [32]:
flatten_dataset_16k_filtered_empty = datasets.load_from_disk("../../data/mt_prepared/manytypes_flat_stacked_no_empty_sv_0k_16k")
np.random.seed(43)
test_ids = np.random.choice(np.arange(len(flatten_dataset_16k_filtered_empty["test"])), 400, replace=False)
test_16k_mt = flatten_dataset_16k_filtered_empty["test"].select(test_ids)
test_16k_mt.save_to_disk("../../data/mt_prepared/test_mt_flat_stacked_no_empty_sv_0k_16k_400")

Saving the dataset (0/1 shards):   0%|          | 0/400 [00:00<?, ? examples/s]

In [32]:
flatten_dataset_32k_filtered_empty = datasets.load_from_disk("../../data/mt_prepared/test_only_manytypes_flat_stacked_no_empty_0k_32k")
np.random.seed(43)
test_ids = np.random.choice(np.arange(len(flatten_dataset_32k_filtered_empty["test"])), 500, replace=False)
test_32k_mt = flatten_dataset_32k_filtered_empty["test"].select(test_ids)
test_32k_mt.save_to_disk("../../data/mt_prepared/test_mt_flat_stacked_no_empty_sv_0k_32k_500")

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

In [43]:
flatten_dataset_32k_filtered_empty = datasets.load_from_disk("../../data/mt_prepared/manytypes_flat_stacked_no_empty_0k_64k")
np.random.seed(43)
flatten_dataset_32k_filtered_empty = flatten_dataset_32k_filtered_empty.filter(lambda example: example["full_len"] < 64*1024)
flatten_dataset_32k_filtered_empty = flatten_dataset_32k_filtered_empty.filter(lambda example: example["full_len"] > 16*1024)
test_ids = np.random.choice(np.arange(len(flatten_dataset_32k_filtered_empty["test"])), 200, replace=False)
test_32k_mt = flatten_dataset_32k_filtered_empty["test"].select(test_ids)
test_32k_mt.save_to_disk("../../data/mt_prepared/test_mt_flat_stacked_no_empty_sv_16k_64k_200")

Filter:   0%|          | 0/18615 [00:00<?, ? examples/s]

Filter:   0%|          | 0/18615 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/200 [00:00<?, ? examples/s]

In [1]:
# merge test data up to 16k and 16k-64k
import datasets
test_16k_mt = datasets.load_from_disk("../../data/mt_prepared/test_mt_flat_stacked_no_empty_sv_0k_16k_400")
test_32k_mt = datasets.load_from_disk("../../data/mt_prepared/test_mt_flat_stacked_no_empty_sv_16k_64k_200")
full_test = datasets.concatenate_datasets([test_16k_mt, test_32k_mt])

In [3]:
full_test.save_to_disk("../../data/mt_prepared/test_mt_flat_stacked_no_empty_sv_0k_64k_600")

Saving the dataset (0/1 shards):   0%|          | 0/600 [00:00<?, ? examples/s]

In [79]:
flatten_dataset_32k_filtered_empty = datasets.load_from_disk("../../data/mt_prepared/manytypes_flat_stacked_no_empty_64k_128k")
np.random.seed(43)
flatten_dataset_32k_filtered_empty = flatten_dataset_32k_filtered_empty.filter(lambda example: example["full_len"] < 96*1024)
flatten_dataset_32k_filtered_empty = flatten_dataset_32k_filtered_empty.filter(lambda example: example["full_len"] > 64*1024)
test_ids = np.random.choice(np.arange(len(flatten_dataset_32k_filtered_empty["test"])), 100, replace=False)
test_32k_mt = flatten_dataset_32k_filtered_empty["test"].select(test_ids)
test_32k_mt.save_to_disk("../../data/mt_prepared/test_mt_flat_stacked_no_empty_sv_64k_96k_100")

Filter:   0%|          | 0/18512 [00:00<?, ? examples/s]

Filter:   0%|          | 0/16822 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

In [83]:
flatten_dataset_32k_filtered_empty = datasets.load_from_disk("../../data/mt_prepared/manytypes_flat_stacked_no_empty_64k_128k")
np.random.seed(43)
flatten_dataset_32k_filtered_empty = flatten_dataset_32k_filtered_empty.filter(lambda example: example["full_len"] < 128*1024)
flatten_dataset_32k_filtered_empty = flatten_dataset_32k_filtered_empty.filter(lambda example: example["full_len"] > 96*1024)
test_ids = np.random.choice(np.arange(len(flatten_dataset_32k_filtered_empty["test"])), 100, replace=False)
test_32k_mt = flatten_dataset_32k_filtered_empty["test"].select(test_ids)
test_32k_mt.save_to_disk("../../data/mt_prepared/test_mt_flat_stacked_no_empty_sv_96k_128k_100")

Filter:   0%|          | 0/18512 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

In [84]:
import datasets
test_16k_mt = datasets.load_from_disk("../../data/mt_prepared/test_mt_flat_stacked_no_empty_sv_0k_16k_400")
test_32k_mt = datasets.load_from_disk("../../data/mt_prepared/test_mt_flat_stacked_no_empty_sv_16k_64k_200")
test_64k_mt = datasets.load_from_disk("../../data/mt_prepared/test_mt_flat_stacked_no_empty_sv_64k_96k_100")
test_96k_mt = datasets.load_from_disk("../../data/mt_prepared/test_mt_flat_stacked_no_empty_sv_96k_128k_100")
full_test = datasets.concatenate_datasets([test_16k_mt, test_32k_mt, test_64k_mt, test_96k_mt])
full_test.save_to_disk("../../data/mt_prepared/test_mt_flat_stacked_no_empty_sv_0k_128k_800")

Saving the dataset (0/1 shards):   0%|          | 0/800 [00:00<?, ? examples/s]

In [82]:
np.percentile(test_32k_mt["length"], 90)

np.float64(104209.5)

In [7]:
!nvidia-smi

Thu Jun 26 12:32:17 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.154.05             Driver Version: 535.154.05   CUDA Version: 12.3     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA H100 80GB HBM3          On  | 00000000:18:00.0 Off |                    0 |
| N/A   45C    P0             392W / 700W |  70791MiB / 81559MiB |     62%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

In [ ]:
flatten_dataset_2k_filtered_empty = datasets.load_from_disk("../../data/mt_prepared/manytypes_flat_stacked_no_empty_0k_4k")


In [6]:
flatten_dataset_2k_filtered_empty["validation"]["answer"][0]

NameError: name 'flatten_dataset_2k_filtered_empty' is not defined

In [ ]:
test_answs = flatten_dataset_2k_filtered_empty["validation"]["answer"]

In [ ]:
unique_answs = np.unique(test_answs, return_counts=True)

In [ ]:
test_answs_ids = np.argsort(unique_answs[1])[::-1][200:]
unique_answs[1][test_answs_ids], unique_answs[0][test_answs_ids]

(array([3, 3, 3, ..., 1, 1, 1], shape=(1304,)),
 array(['typing.Optional[builtins.bytes]',
        'typing.List[typing.Union[typing.Dict[builtins.str, builtins.str], typing.Tuple[builtins.str, builtins.str, builtins.int, builtins.bool]]]',
        'wemake_python_styleguide.visitors.ast.loops.WrongComprehensionVisitor',
        ..., 'UserStatus', 'aiko.utils.ProxyAttr',
        'aiko.application.Application'], shape=(1304,), dtype='<U661'))

In [108]:
t11 = flatten_dataset_2k["train"].select(list(range(10)))

In [111]:
t11 = t11.map(prepare_text)
t11[1]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

{'repo_name': '007MrNiko/Stepper',
 'fname': ['repos/007MrNiko/Stepper/manage.py',
  'repos/007MrNiko/Stepper/accounts/urls.py',
  'repos/007MrNiko/Stepper/accounts/admin.py',
  'repos/007MrNiko/Stepper/accounts/models.py',
  'repos/007MrNiko/Stepper/accounts/tests.py',
  'repos/007MrNiko/Stepper/accounts/views.py'],
 'code': '[comment] \nimport os \nimport sys \n\nif __name__ == [string] : \nos . environ . setdefault ( [string] , [string] ) \n\nfrom django . core . management import execute_from_command_line \n\nexecute_from_command_line ( sys . argv ) \n\nfrom typing import List , Any \nimport typing \nfrom django . urls import path \n\nfrom accounts import views \n\n[comment] \napp_name = [string] \nurlpatterns = [ path ( [string] , views . login_view , name = [string] ) , path ( [string] , views . logout_view , name = [string] ) , path ( [string] , views . register , name = [string] ) , ] \n\nfrom django . contrib import admin \n\n[comment] \n\nfrom django . db import models \n\n[c

In [93]:
fl_ds

Dataset({
    features: ['repo_name', 'fname', 'code', 'variables', 'length', 'answer'],
    num_rows: 233930
})

In [62]:
small_code_repos_2k = code_repos_2k.select(list(range(100)))

In [87]:
def flatten_samples(example):
    prompt_template = lambda variable: f"What is the type of variable {variable}?"
    new_examples = {key: [] for key in example.keys()}#copy.deepcopy(example)
    new_examples["answer"] = []
    for v in example["variables"]:
        for key in new_examples.keys():
            if key == "variables":
                new_examples[key].append(prompt_template(v[0]))
                new_examples["answer"].append(v[1])
            elif key == "answer":
                pass
            else:
                new_examples[key].append(example[key])
    return new_examples

In [88]:
multi_code_repos_2k = small_code_repos_2k.map(flatten_samples)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [89]:
flat_code_repos_2k = {key: [] for key in multi_code_repos_2k[0].keys()}
for el in multi_code_repos_2k:
    for key in flat_code_repos_2k.keys():
        flat_code_repos_2k[key] += el[key]

In [90]:
fl_ds = datasets.Dataset.from_dict(flat_code_repos_2k)

In [91]:
fl_ds[0]

{'repo_name': '007MrNiko/Stepper',
 'fname': ['repos/007MrNiko/Stepper/manage.py',
  'repos/007MrNiko/Stepper/accounts/urls.py'],
 'code': '[comment] \nimport os \nimport sys \n\nif __name__ == [string] : \nos . environ . setdefault ( [string] , [string] ) \n\nfrom django . core . management import execute_from_command_line \n\nexecute_from_command_line ( sys . argv ) \n\nfrom typing import List , Any \nimport typing \nfrom django . urls import path \n\nfrom accounts import views \n\n[comment] \napp_name = [string] \nurlpatterns = [ path ( [string] , views . login_view , name = [string] ) , path ( [string] , views . logout_view , name = [string] ) , path ( [string] , views . register , name = [string] ) , ] \n',
 'variables': 'What is the type of variable urlpatterns?',
 'length': 209,
 'answer': 'typing.List[typing.Any]'}

In [81]:
small_code_repos_2k[0]

{'repo_name': '007MrNiko/Stepper',
 'fname': ['repos/007MrNiko/Stepper/manage.py',
  'repos/007MrNiko/Stepper/accounts/urls.py'],
 'code': '[comment] \nimport os \nimport sys \n\nif __name__ == [string] : \nos . environ . setdefault ( [string] , [string] ) \n\nfrom django . core . management import execute_from_command_line \n\nexecute_from_command_line ( sys . argv ) \n\nfrom typing import List , Any \nimport typing \nfrom django . urls import path \n\nfrom accounts import views \n\n[comment] \napp_name = [string] \nurlpatterns = [ path ( [string] , views . login_view , name = [string] ) , path ( [string] , views . logout_view , name = [string] ) , path ( [string] , views . register , name = [string] ) , ] \n',
 'variables': [['urlpatterns', 'typing.List[typing.Any]']],
 'length': 209}

In [77]:
n1[1]

{'repo_name': ['007MrNiko/Stepper', '007MrNiko/Stepper'],
 'fname': [['repos/007MrNiko/Stepper/accounts/views.py'],
  ['repos/007MrNiko/Stepper/accounts/views.py']],
 'code': ['from typing import Any \nimport accounts \nimport typing \nfrom django . shortcuts import render , redirect \nfrom django . contrib . auth import authenticate , login , logout \nfrom django . contrib . auth . models import User \n\nfrom accounts . forms import LoginForm , RegistrationForm \nfrom lists . forms import TodoForm \n\n[comment] \ndef login_view ( request ) : \nif request . method == [string] : \nform = LoginForm ( request . POST ) \nif form . is_valid ( ) : \nuser = authenticate ( username = request . POST [ [string] ] , password = request . POST [ [string] ] ) \nif user is not None : \nif user . is_active : \nlogin ( request , user ) \nreturn redirect ( [string] ) \nelse : \nreturn render ( request , [string] , { [string] : form } ) \nelse : \nreturn render ( request , [string] , { [string] : LoginFo

In [59]:
code_repos_2k_f = code_repos_2k.flatten()

In [60]:
code_repos_2k_f

Dataset({
    features: ['repo_name', 'fname', 'code', 'variables', 'length'],
    num_rows: 30504
})

In [58]:
code_repos_2k[2]

{'repo_name': '007MrNiko/Stepper',
 'fname': ['repos/007MrNiko/Stepper/accounts/forms.py'],
 'code': 'from typing import Type , Any \nimport accounts \nimport typing \nfrom django import forms \nfrom django . contrib . auth . models import User \n\n\ndef widget_attrs ( placeholder ) : \nreturn { [string] : [string] , [string] : placeholder } \n\n\ndef form_kwargs ( widget , label = [string] , max_length = [number] ) : \nreturn { [string] : widget , [string] : label , [string] : max_length } \n\n[comment] \nclass LoginForm ( forms . Form ) : \n\nusername = forms . CharField ( ** form_kwargs ( widget = forms . TextInput ( attrs = widget_attrs ( [string] ) ) ) ) \npassword = forms . CharField ( ** form_kwargs ( widget = forms . PasswordInput ( attrs = widget_attrs ( [string] ) ) ) ) \n\n[comment] \ndef clean ( self ) : \n[comment] \nif self . errors : \nreturn self . cleaned_data \n\nusername = self . cleaned_data . get ( [string] ) \npassword = self . cleaned_data . get ( [string] ) \nus

In [31]:
tr_lens = code_repos_2k["length"]
np.mean(tr_lens), np.percentile(tr_lens, 10), np.percentile(tr_lens, 20), np.percentile(tr_lens, 80), np.percentile(tr_lens, 95)

(np.float64(2558.9838512399624),
 np.float64(334.0),
 np.float64(588.0),
 np.float64(2648.0),
 np.float64(7563.799999999988))

In [63]:
new_repos[465:467]

[Dataset({
     features: ['repo_name', 'fname', 'code', 'variables', 'length'],
     num_rows: 1
 }),
 Dataset({
     features: ['repo_name', 'fname', 'code', 'variables', 'length'],
     num_rows: 2
 })]

In [72]:
new_repos[465][0]

{'repo_name': 'MartinSoto/aiohttp-fastproxy',
 'fname': ['repos/MartinSoto/aiohttp-fastproxy/tests/test_fastproxy.py',
  'repos/MartinSoto/aiohttp-fastproxy/tests/utils.py',
  'repos/MartinSoto/aiohttp-fastproxy/tests/conftest.py',
  'repos/MartinSoto/aiohttp-fastproxy/fastproxy/fastproxy.py'],
 'code': 'import pytest \n\npytestmark = pytest . mark . asyncio \n\n\nasync def test_simple_get ( downstream_app , proxy_app , http_client_session ) : \nasync with http_client_session . get ( proxy_app . url ) as resp : \nassert downstream_app . request . method == [string] \nassert downstream_app . request . path == [string] \nassert downstream_app . body == [string] \nassert await resp . text ( ) == [string] \n\n\nasync def test_path_get ( downstream_app , proxy_app , http_client_session ) : \nasync with http_client_session . get ( proxy_app . url / [string] ) as resp : \nassert downstream_app . request . method == [string] \nassert downstream_app . request . path == [string] \nassert downstr

In [74]:
new_repos[466][1]

{'repo_name': 'Marto32/sentinel',
 'fname': ['repos/Marto32/sentinel/util/twilio_api_helper.py'],
 'code': 'from typing import Dict \nimport typing \nimport twilio \nimport builtins \nfrom twilio . rest import Client \nfrom settings import ( TWILIO_ACCOUNT_SID , TWILIO_AUTH_TOKEN , TWILIO_SOURCE_PHONE , TWILIO_DESTINATION_PHONE ) \n\n\nclass SendNotification ( object ) : \n[docstring] \n\ndef __init__ ( self , message ) : \n[comment] \n[docstring] \nself . message_body = message \nself . client = Client ( TWILIO_ACCOUNT_SID , TWILIO_AUTH_TOKEN ) \n\n@ property def payload ( self ) : \nparams = { [string] : self . message_body , [string] : TWILIO_DESTINATION_PHONE , [string] : TWILIO_SOURCE_PHONE , } \nreturn params \n\ndef emit ( self ) : \n[docstring] \nreturn self . client . messages . create ( ** self . payload ) \n\n',
 'variables': [],
 'length': 217}

In [75]:
new_repos[466].cast(new_repos[465].features)

Casting the dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Dataset({
    features: ['repo_name', 'fname', 'code', 'variables', 'length'],
    num_rows: 2
})

In [65]:
new_repos[465].features

{'repo_name': Value(dtype='string', id=None),
 'fname': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
 'code': Value(dtype='string', id=None),
 'variables': Sequence(feature=Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), length=-1, id=None),
 'length': Value(dtype='int64', id=None)}

In [66]:
new_repos[466].features

{'repo_name': Value(dtype='string', id=None),
 'fname': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
 'code': Value(dtype='string', id=None),
 'variables': Sequence(feature=Value(dtype='null', id=None), length=-1, id=None),
 'length': Value(dtype='int64', id=None)}

In [33]:
unique_repos = list(set(manytypes_ds["train"]["repo_name"]))
filtered_repos = []
new_repos = []

for unique_repo in tqdm(unique_repos[:20]):
    repo_1 = manytypes_ds["train"].filter(lambda example: example["repo_name"] == unique_repo)
    filtered_repos.append(repo_1)
for f_repo in tqdm(filtered_repos):
    new_repo_1 = stack_one_repo(f_repo, 0, 2048)
    new_repos.append(new_repo_1)

  0%|          | 0/20 [00:00<?, ?it/s]

Filter:   0%|          | 0/141525 [00:00<?, ? examples/s]

 55%|█████▌    | 11/20 [00:02<00:01,  4.91it/s]

Filter:   0%|          | 0/141525 [00:00<?, ? examples/s]

 60%|██████    | 12/20 [00:04<00:03,  2.14it/s]

...

 90%|█████████ | 18/20 [00:19<00:03,  1.86s/it]

Filter:   0%|          | 0/141525 [00:00<?, ? examples/s]

 95%|█████████▌| 19/20 [00:21<00:02,  2.04s/it]

Filter:   0%|          | 0/141525 [00:00<?, ? examples/s]

100%|██████████| 20/20 [00:00<00:00, 61.43it/s]


In [32]:
new_repos

[Dataset({
     features: ['repo_name', 'fname', 'code', 'variables', 'length'],
     num_rows: 15
 }),
 Dataset({
     features: ['repo_name', 'fname', 'code', 'variables', 'length'],
     num_rows: 17
 }),
 Dataset({
     features: ['repo_name', 'fname', 'code', 'variables', 'length'],
     num_rows: 3
 }),
 Dataset({
     features: ['repo_name', 'fname', 'code', 'variables', 'length'],
     num_rows: 6
 }),
 Dataset({
     features: ['repo_name', 'fname', 'code', 'variables', 'length'],
     num_rows: 1
 }),
 Dataset({
     features: ['repo_name', 'fname', 'code', 'variables', 'length'],
     num_rows: 1
 }),
 Dataset({
     features: ['repo_name', 'fname', 'code', 'variables', 'length'],
     num_rows: 2
 }),
 Dataset({
     features: ['repo_name', 'fname', 'code', 'variables', 'length'],
     num_rows: 19
 }),
 Dataset({
     features: ['repo_name', 'fname', 'code', 'variables', 'length'],
     num_rows: 5
 }),
 Dataset({
     features: ['repo_name', 'fname', 'code', 'variables', 

In [28]:
nr_1 = stack_one_repo(repo_1, 0, 2048)

In [25]:
# on each step, randomly choose length below threshold
len_thr = 2048
np.random.seed(42)
curr_thr = np.random.randint(0, len_thr)
# after choose samples up to curr_thr and concatenate them
idx = 0
new_repo_1 = {
    "repo_name": [],
    "fname": [],
    "code": [],
    "variables": [],
    "length": [],
}
while idx < len(repo_1):
    curr_len = repo_1[idx]["length"]
    curr_repo = repo_1[idx]["repo_name"]
    curr_fnames = [repo_1[idx]["fname"]]
    curr_code = repo_1[idx]["code"]
    curr_vars = repo_1[idx]["variables"]
    while curr_len < curr_thr and idx+1 < len(repo_1):
        # iterate over ids to concate text
        idx += 1
        curr_len += repo_1[idx]["length"]
        curr_fnames += [repo_1[idx]["fname"]]
        curr_code += "\n" + repo_1[idx]["code"]
        curr_vars += repo_1[idx]["variables"]
    idx += 1
    new_repo_1["repo_name"].append(curr_repo)
    new_repo_1["fname"].append(curr_fnames)
    new_repo_1["code"].append(curr_code)
    new_repo_1["variables"].append(curr_vars)
    new_repo_1["length"].append(curr_len)
    print(idx)


4
8
11
14
18
23
28


In [26]:
new_repo_1

{'repo_name': ['LayerXcom/cbc-casper-simulator',
  'LayerXcom/cbc-casper-simulator',
  'LayerXcom/cbc-casper-simulator',
  'LayerXcom/cbc-casper-simulator',
  'LayerXcom/cbc-casper-simulator',
  'LayerXcom/cbc-casper-simulator',
  'LayerXcom/cbc-casper-simulator'],
 'fname': [['repos/LayerXcom/cbc-casper-simulator/main.py',
   'repos/LayerXcom/cbc-casper-simulator/cbc_casper_simulator/validator.py',
   'repos/LayerXcom/cbc-casper-simulator/cbc_casper_simulator/justification.py',
   'repos/LayerXcom/cbc-casper-simulator/cbc_casper_simulator/state.py'],
  ['repos/LayerXcom/cbc-casper-simulator/cbc_casper_simulator/block_proposer.py',
   'repos/LayerXcom/cbc-casper-simulator/cbc_casper_simulator/error.py',
   'repos/LayerXcom/cbc-casper-simulator/cbc_casper_simulator/message.py',
   'repos/LayerXcom/cbc-casper-simulator/cbc_casper_simulator/message_validator.py'],
  ['repos/LayerXcom/cbc-casper-simulator/cbc_casper_simulator/block.py',
   'repos/LayerXcom/cbc-casper-simulator/cbc_casper_s

In [24]:
repo_1

Dataset({
    features: ['repo_name', 'fname', 'code', 'variables', 'length'],
    num_rows: 28
})

In [18]:
np.sum(repo_1["length"])

np.int64(9345)

In [19]:
repo_1

Dataset({
    features: ['repo_name', 'fname', 'code', 'variables', 'length'],
    num_rows: 28
})

In [17]:
repo_1["length"]

[148,
 610,
 168,
 463,
 324,
 158,
 379,
 285,
 295,
 394,
 1016,
 93,
 560,
 691,
 205,
 99,
 600,
 646,
 241,
 249,
 105,
 173,
 636,
 95,
 149,
 447,
 57,
 59]

In [89]:
t1 = caclulate_length(manytypes_ds["train"][4])
len(t1["variables"])
t1

{'repo_name': 'improbable-research/keanu',
 'fname': 'repos/improbable-research/keanu/keanu-python/tests/test_java_exception.py',
 'code': 'from typing import Any \nimport typing \nimport pytest \nfrom py4j . protocol import Py4JJavaError \n\nfrom keanu . context import KeanuContext \nfrom keanu . java_exception import JavaException \n\n\ndef test_you_can_get_info_from_a_java_exception ( ) : \ncontext = KeanuContext ( ) \nwith pytest . raises ( Py4JJavaError ) as excinfo : \ncontext . jvm_view ( ) . java . util . HashMap ( - [number] ) \n\njava_exception = JavaException ( excinfo . value ) \nassert java_exception . get_class ( ) == [string] \nassert java_exception . get_message ( ) == [string] \nassert java_exception . unwrap ( ) . getCause ( ) == None \n\n\ndef test_you_can_throw_a_java_exception ( ) : \nwith pytest . raises ( JavaException , match = [string] ) : \ncontext = KeanuContext ( ) \ntry : \ncontext . jvm_view ( ) . java . util . HashMap ( - [number] ) \nexcept Py4JJavaError

In [17]:
len(t1["variables"].keys())

22737

In [ ]:
proc_data["train"] = train_ds

In [ ]:
train_ds.save_to_disk("../data/manytypes_raw")

In [49]:
train_ds

Dataset({
    features: ['repo_name', 'fname', 'code', 'variables'],
    num_rows: 141525
})

In [24]:
repo = train_repos[0]
res = extract_repo_data(path, repo)

In [27]:
idx = 1
res["variables"][idx], res["code"][idx]

({'gaussian': 'typing.Any',
  'sample': 'typing.Any',
  'mu': 'typing.Any',
  'ndarray': 'typing.Any',
  'vertex': 'typing.Any',
  'value': 'typing.Any',
  'scalar': 'builtins.float',
  'sigma': 'typing.Any',
  'label': 'builtins.str',
  'parent': 'typing.Tuple[typing.Any, typing.Any]',
  'child': 'typing.Tuple[typing.Any, ...]'},
 'from typing import Callable , Type , Union , Any , List , Tuple , Set [EOL] import keanu [EOL] import builtins [EOL] import typing [EOL] import py4j [EOL] from typing import Type , Callable , Tuple , Union [EOL] [EOL] import numpy as np [EOL] import pandas as pd [EOL] import pytest [EOL] from py4j . java_gateway import JVMView [EOL] [EOL] from keanu import set_deterministic_state [EOL] from keanu . context import KeanuContext [EOL] from keanu . vartypes import tensor_arg_types , primitive_types , numpy_types , pandas_types [EOL] from keanu . vertex import Gaussian , Const , UniformInt , Bernoulli , IntegerProxy , Double [EOL] from keanu . vertex . base impo

In [28]:
print(res["code"][1].replace("[EOL] ", "\n").replace("[EOL]", "\n"))

from typing import Callable , Type , Union , Any , List , Tuple , Set 
import keanu 
import builtins 
import typing 
import py4j 
from typing import Type , Callable , Tuple , Union 

import numpy as np 
import pandas as pd 
import pytest 
from py4j . java_gateway import JVMView 

from keanu import set_deterministic_state 
from keanu . context import KeanuContext 
from keanu . vartypes import tensor_arg_types , primitive_types , numpy_types , pandas_types 
from keanu . vertex import Gaussian , Const , UniformInt , Bernoulli , IntegerProxy , Double 
from keanu . vertex . base import Vertex 


@ pytest . fixture def jvm_view ( ) : 
from py4j . java_gateway import java_import 
jvm_view = KeanuContext ( ) . jvm_view ( ) 
java_import ( jvm_view , [string] ) 
return jvm_view 


def assert_vertex_value_equals_scalar ( vertex , expected_type , scalar ) : 
vertex_value = vertex . get_value ( ) 
assert vertex_value == scalar 
assert type ( vertex_value ) == numpy_types 
assert vertex_value . shap

In [14]:
# debug
repo = train_repos[0]
repo_path = os.path.join(path, "processed_projects_clean", repo)
# load repo code
with open(repo_path, "rb") as f:
    repo_code = json.load(f)
repo_name = list(repo_code.keys())[0] # check if all repos have one key
repo_files = list(repo_code[repo_name]['src_files'].keys())

In [15]:
repo_code.keys()

dict_keys(['improbable-research/keanu'])

In [16]:
repo_code['improbable-research/keanu']['src_files']['repos/improbable-research/keanu/keanu-project/src/test/resources/model.py'].keys()

dict_keys(['untyped_seq', 'typed_seq', 'imports', 'variables', 'mod_var_occur', 'mod_var_ln', 'classes', 'funcs', 'set', 'tc', 'no_types_annot', 'type_annot_cove'])

In [17]:
seq_no_type = repo_code[repo_name]['src_files'][repo_files[1]]["untyped_seq"]
seq_types = repo_code[repo_name]['src_files'][repo_files[1]]["typed_seq"]
seq_vars = repo_code[repo_name]['src_files'][repo_files[1]]["variables"]

In [18]:
repo_code[repo_name]['src_files'][repo_files[1]].keys()

dict_keys(['untyped_seq', 'typed_seq', 'imports', 'variables', 'mod_var_occur', 'mod_var_ln', 'classes', 'funcs', 'set', 'tc', 'no_types_annot', 'type_annot_cove'])

In [22]:
repo_code[repo_name]['src_files'][repo_files[1]]["classes"]

[]

In [20]:
print(seq_no_type.replace("[EOL] ", "\n").replace("[EOL]", "\n"))

from typing import Any , Dict 
import builtins 
import typing 
from setuptools import setup , find_packages 
import shutil 
import os 


def readme ( ) : 
with open ( [string] , [string] ) as f : 
return f . read ( ) 


here = os . path . abspath ( os . path . dirname ( __file__ ) ) 
about = { } 
with open ( os . path . join ( here , [string] , [string] ) , [string] , encoding = [string] ) as f : 
exec ( f . read ( ) , about ) 

[comment] 
for dir_name in ( [string] % about [ [string] ] , [string] , [string] , [string] ) : 
shutil . rmtree ( dir_name , ignore_errors = True ) 

setup ( name = [string] , description = [string] , long_description = readme ( ) , version = about [ [string] ] , author = [string] , author_email = [string] , url = [string] , license = [string] , packages = find_packages ( exclude = [ [string] ] ) , include_package_data = True , install_requires = [ [string] , [string] , [string] , [string] ] , classifiers = [ [string] , [string] , [string] ] , zip_safe = False

In [56]:
seq_types

'0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 $builtins.str$ 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 $builtins.str$ 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 $typing.Dict[typing.Any,typing.Any]$ 0 0 0 0 0 0 0 0 0 0 0 0 0 $builtins.str$ 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 $typing.Dict[typing.Any,typing.Any]$ 0 0 0 0 0 0 0 0 0 0 0 $typing.Dict[typing.Any,typing.Any]$ 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 $typing.Dict[typing.Any,typing.Any]$ 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0'

In [54]:
print(seq_vars)

{'here': 'builtins.str', 'about': 'typing.Dict[typing.Any, typing.Any]'}


In [18]:
repos[:10]

['dtgoitiatelegram-bot-test.json',
 'mabjardelezaprecision-farming-ph.json',
 'potykiondoctest2code.json',
 'anothergitdevaws-lambda-ci-python.json',
 'impredicativegitblobts.json',
 'webrecorderbrowsertrix.json',
 'arraiyopensourcekornia.json',
 'jettifydclock.json',
 'n8henriesimplenet.json',
 'needlehaystackneedlestack.json']

In [19]:
len(repos)

5093

In [20]:
sample_path = "../../data/ManyTypes4PyDataset-v0.7/processed_projects_clean/sim1234snowid.json"

In [21]:
with open(sample_path, "rb") as f:
    sample = json.load(f)

In [22]:
source_code = sample['sim1234/snowid']['src_files']['repos/sim1234/snowid/project/game.py']['untyped_seq']

In [23]:
code = source_code.replace("[EOL] ", "\n").replace("[EOL]", "\n")

In [25]:
print(code)

from typing import Any , Type 
import typing 
import project 
import os 

import sdl2 
import sdl2 . ext 

from gamepart import Game 

from context import MyContext 


ROOT = os . path . dirname ( os . path . abspath ( __file__ ) ) 
RESOURCES = os . path . join ( ROOT , [string] ) 


class MyGame ( Game ) : 
context_class = MyContext 

def get_config ( self ) : 
config = super ( ) . get_config ( ) 
config [ [string] ] = [number] 
[comment] 
return config 

def init_font_manager ( self ) : 
self . font_manager = sdl2 . ext . FontManager ( os . path . join ( RESOURCES , [string] ) ) 

def init_heavy ( self ) : 
self . font_manager . add ( os . path . join ( RESOURCES , [string] ) , [string] ) 

def init_scenes ( self ) : 
import scenes 

self . add_scene ( [string] , scenes . TestScene ) 
self . add_scene ( [string] , scenes . BallScene ) 
super ( ) . init_scenes ( ) 
self . queue_scene_switch ( [string] ) 



# EDA for ManyTypes4Py

In [2]:
path = "../../data/ManyTypes4PyDataset-v0.7/"

In [3]:
df = pd.read_csv(path + "dataset_split.csv")

In [4]:
df

,train,repos/HackPartners/shush/setup.py
0,train,repos/HackPartners/shush/shush/client.py
1,train,repos/HackPartners/shush/shush/__init__.py
2,train,repos/tedle/uitabot/bot/uitabot.py
3,train,repos/tedle/uitabot/bot/test/test_audio.py
4,train,repos/tedle/uitabot/bot/test/test_youtube_api.py
...,...,...
293664,valid,repos/binaryshao/ST3config/Packages/Python PEP...
293665,valid,repos/binaryshao/ST3config/Packages/ConvertToU...
293666,valid,repos/binaryshao/ST3config/Packages/Anaconda/a...
293667,valid,repos/binaryshao/ST3config/Packages/Anaconda/a...


In [5]:
sample_path = "../../data/ManyTypes4PyDataset-v0.7/processed_projects_clean/sim1234snowid.json"

In [6]:
with open(sample_path, "rb") as f:
    sample = json.load(f)

In [19]:
source_code = sample['sim1234/snowid']['src_files']['repos/sim1234/snowid/project/game.py']['untyped_seq']

In [20]:
code = source_code.replace("[EOL] ", "\n").replace("[EOL]", "\n")

In [21]:
print(code)

from typing import Any , Type 
import typing 
import project 
import os 

import sdl2 
import sdl2 . ext 

from gamepart import Game 

from context import MyContext 


ROOT = os . path . dirname ( os . path . abspath ( __file__ ) ) 
RESOURCES = os . path . join ( ROOT , [string] ) 


class MyGame ( Game ) : 
context_class = MyContext 

def get_config ( self ) : 
config = super ( ) . get_config ( ) 
config [ [string] ] = [number] 
[comment] 
return config 

def init_font_manager ( self ) : 
self . font_manager = sdl2 . ext . FontManager ( os . path . join ( RESOURCES , [string] ) ) 

def init_heavy ( self ) : 
self . font_manager . add ( os . path . join ( RESOURCES , [string] ) , [string] ) 

def init_scenes ( self ) : 
import scenes 

self . add_scene ( [string] , scenes . TestScene ) 
self . add_scene ( [string] , scenes . BallScene ) 
super ( ) . init_scenes ( ) 
self . queue_scene_switch ( [string] ) 



In [22]:
len(source_code.split(" "))

220

In [23]:
source_types = sample['sim1234/snowid']['src_files']['repos/sim1234/snowid/project/game.py']['typed_seq']

In [31]:
code_types = source_types.split(" ")
for el, typ in zip(source_code.split(" "), source_types.split(" ")):
    if typ != "0":
        print(el, typ)

ROOT $builtins.str$
RESOURCES $builtins.str$
ROOT $builtins.str$
context_class $typing.Type[project.game.MyGame]$
config $typing.Any$
config $typing.Any$
config $typing.Any$
RESOURCES $builtins.str$
RESOURCES $builtins.str$


In [9]:
sample

{'sim1234/snowid': {'src_files': {'repos/sim1234/snowid/project/game.py': {'untyped_seq': 'from typing import Any , Type [EOL] import typing [EOL] import project [EOL] import os [EOL] [EOL] import sdl2 [EOL] import sdl2 . ext [EOL] [EOL] from gamepart import Game [EOL] [EOL] from context import MyContext [EOL] [EOL] [EOL] ROOT = os . path . dirname ( os . path . abspath ( __file__ ) ) [EOL] RESOURCES = os . path . join ( ROOT , [string] ) [EOL] [EOL] [EOL] class MyGame ( Game ) : [EOL] context_class = MyContext [EOL] [EOL] def get_config ( self ) : [EOL] config = super ( ) . get_config ( ) [EOL] config [ [string] ] = [number] [EOL] [comment] [EOL] return config [EOL] [EOL] def init_font_manager ( self ) : [EOL] self . font_manager = sdl2 . ext . FontManager ( os . path . join ( RESOURCES , [string] ) ) [EOL] [EOL] def init_heavy ( self ) : [EOL] self . font_manager . add ( os . path . join ( RESOURCES , [string] ) , [string] ) [EOL] [EOL] def init_scenes ( self ) : [EOL] import scenes 

# CUAD

In [3]:
# Prepare CUAD
# Check lengths distribution
# split by segments - from 1-2 to 8-16
# save splits

In [4]:
cuad = datasets.load_dataset("theatticusproject/cuad-qa")

Using the latest cached version of the module from /home/jovyan/.cache/huggingface/modules/datasets_modules/datasets/theatticusproject--cuad-qa/01ed7dc61ab84230462731422e77cbb6f54ea8590b22a2d881b594f4d7f3746c (last modified on Fri Apr 11 18:41:35 2025) since it couldn't be found locally at theatticusproject/cuad-qa, or remotely on the Hugging Face Hub.


In [5]:
cuad

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 22450
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 4182
    })
})

In [6]:
tokenizer = AutoTokenizer.from_pretrained("unsloth/Llama-3.2-1B-Instruct")

In [42]:
def build_split_by_length(dataset, min_length=0, max_length=2048, val_percentage=0.1, filter_ns=False):
    if filter_ns:
        buf_data = dataset.filter(lambda example: len(example['answers']['text']) != 0)
        buf_data = buf_data.filter(lambda example: np.logical_and(len(example["tokens"]) >= min_length, len(example["tokens"]) < max_length))
    else:
        # filter samples by length
        buf_data = dataset.filter(lambda example: np.logical_and(len(example["tokens"]) >= min_length, len(example["tokens"]) < max_length))
    # split valdation from train, but check that context does not overlap between train and val
    texts, counts = np.unique(buf_data["train"]["context"], return_counts=True)
    val_size = int(val_percentage * len(buf_data["train"]))
    val_len = 0
    val_texts = []
    for idx in range(len(texts)):
        val_texts.append(texts[idx])
        val_len += counts[idx]
        if val_len >= val_size:
            break
    val_data = buf_data["train"].filter(lambda example: example["context"] in val_texts)
    train_data = buf_data["train"].filter(lambda example: example["context"] not in val_texts)
    buf_data["validation"] = val_data
    buf_data["train"] = train_data
    return buf_data

In [8]:
# tokenize texts to calc length
def tokenize(el):
    if len(el['answers']['text']) != 0:
        answer = el['answers']['text'][0]
    else:
        answer = "Not stated."
    messages = [
        {"role": "user", "content": f"Question:\n{el['question']}\nContext:\n{el['context']}\n"},
        {"role": "assistant", "content": f"Answer:\n{answer}\n"},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )
    return {"tokens": text}

In [9]:
cuad["train"][11]["answers"]["text"]

[]

In [10]:
tok_cuad = cuad.map(tokenize)

In [16]:
b1 = build_split_by_length(tok_cuad)

Filter:   0%|          | 0/2314 [00:00<?, ? examples/s]

In [17]:
b1

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
        num_rows: 2050
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
        num_rows: 656
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
        num_rows: 264
    })
})

In [18]:
b1.save_to_disk("../../data/cuad_prepared/cuad_0k_2k")

Saving the dataset (0/1 shards):   0%|          | 0/2050 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/656 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/264 [00:00<?, ? examples/s]

In [20]:
start_size = [0, 2048, 4096, 8192, 16384]
end_size = [2048, 4096, 8192, 16384, 32768]
for idx in range(len(start_size)):
    buf_data = build_split_by_length(tok_cuad, start_size[idx], end_size[idx])
    save_path = f"../../data/cuad_prepared/cuad_{start_size[idx]//1000}k_{end_size[idx]//1000}k"
    print(save_path)
    buf_data.save_to_disk(save_path)

../../data/cuad_prepared/cuad_0k_2k


Saving the dataset (0/1 shards):   0%|          | 0/2050 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/656 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/264 [00:00<?, ? examples/s]

Filter:   0%|          | 0/22450 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/3922 [00:00<?, ? examples/s]

../../data/cuad_prepared/cuad_16k_32k


Saving the dataset (0/2 shards):   0%|          | 0/3469 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/494 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/453 [00:00<?, ? examples/s]

In [21]:
start_size = [0, 0, 0, 0, 0]
end_size = [2048, 4096, 8192, 16384, 32768]
for idx in range(len(start_size)):
    buf_data = build_split_by_length(tok_cuad, start_size[idx], end_size[idx])
    save_path = f"../../data/cuad_prepared/cuad_{start_size[idx]//1000}k_{end_size[idx]//1000}k"
    print(save_path)
    buf_data.save_to_disk(save_path)

../../data/cuad_prepared/cuad_0k_2k


Saving the dataset (0/1 shards):   0%|          | 0/2050 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/656 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/264 [00:00<?, ? examples/s]

Filter:   0%|          | 0/22450 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/20345 [00:00<?, ? examples/s]

../../data/cuad_prepared/cuad_0k_32k


Saving the dataset (0/7 shards):   0%|          | 0/18276 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/3977 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2069 [00:00<?, ? examples/s]

In [ ]:
# TODO: maybe build datasets for 3 and 5 segments instead of 4s

In [43]:
# Now filter all ns samples to more real dataset
start_size = [0, 2048, 4096, 8192, 16384]
end_size = [2048, 4096, 8192, 16384, 32768]
for idx in range(len(start_size)):
    buf_data = build_split_by_length(tok_cuad, start_size[idx], end_size[idx], filter_ns=True)
    save_path = f"../../data/cuad_prepared/cuad_no_ns_{start_size[idx]//1000}k_{end_size[idx]//1000}k"
    print(save_path)
    buf_data.save_to_disk(save_path)

Filter:   0%|          | 0/11180 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1244 [00:00<?, ? examples/s]

Filter:   0%|          | 0/470 [00:00<?, ? examples/s]

Filter:   0%|          | 0/470 [00:00<?, ? examples/s]

../../data/cuad_prepared/cuad_no_ns_0k_2k


...

Filter:   0%|          | 0/2605 [00:00<?, ? examples/s]

../../data/cuad_prepared/cuad_no_ns_16k_32k


Saving the dataset (0/1 shards):   0%|          | 0/2281 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/228 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/324 [00:00<?, ? examples/s]

In [44]:
d1 = datasets.load_from_disk("../../data/cuad_prepared/cuad_no_ns_0k_2k")

In [45]:
len(d1["train"]["answers"][5]["text"])

1

In [46]:
start_size = [0, 0, 0, 0, 0]
end_size = [2048, 4096, 8192, 16384, 32768]
for idx in range(len(start_size)):
    buf_data = build_split_by_length(tok_cuad, start_size[idx], end_size[idx], filter_ns=True)
    save_path = f"../../data/cuad_prepared/cuad_no_ns_{start_size[idx]//1000}k_{end_size[idx]//1000}k"
    print(save_path)
    buf_data.save_to_disk(save_path)

../../data/cuad_prepared/cuad_no_ns_0k_2k


Saving the dataset (0/1 shards):   0%|          | 0/421 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/71 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/49 [00:00<?, ? examples/s]

Filter:   0%|          | 0/11180 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/9573 [00:00<?, ? examples/s]

../../data/cuad_prepared/cuad_no_ns_0k_32k


Saving the dataset (0/3 shards):   0%|          | 0/8610 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1129 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/963 [00:00<?, ? examples/s]

In [47]:
buf_data

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
        num_rows: 8610
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
        num_rows: 1129
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
        num_rows: 963
    })
})

# Now generate additional samples for CUAD

In [286]:
from tqdm import tqdm
import copy

In [309]:
# enhance 2k split 
def enlarge_split_by_length(dataset, pipe, min_length=0, max_length=2048, prompt=None, multiplication_ratio=1,
                            filter_ns=False):
    base_seed = 42
    set_seed(base_seed)
    print(len(dataset["train"]))
    if filter_ns:
        buf_data = dataset.filter(lambda example: len(example['answers']['text']) != 0)
        buf_data = buf_data.filter(lambda example: np.logical_and(len(example["tokens"]) >= min_length, len(example["tokens"]) < max_length))
    else:
        # filter samples by length
        buf_data = dataset.filter(lambda example: np.logical_and(len(example["tokens"]) >= min_length, len(example["tokens"]) < max_length))
    # split valdation from train, but check that context does not overlap between train and val
    print(len(buf_data["train"]))
    generated_pairs = {
        'id': [],
        'title': [],
        'context': [],
        'question': [],
        'answers': [],
    }
    # select only unique context
    unique_data_ids = []
    unique_titles = []
    for el in dataset["train"]:
        if el["title"] not in unique_titles:
            unique_titles.append(el["title"])
            unique_data_ids.append(el["id"])
    buf_data = buf_data["train"].filter(lambda example: example["id"] in unique_data_ids)
    print(len(buf_data))
    for el in tqdm(buf_data):
        buf_questions = []
        #while ratio_count < multiplication_ratio and trials < multiplication_ratio * 2:
        #set_seed(base_seed+trials)
        #trials += 1
        # generate qa
        messages = [
            {"role": "system", "content": "You are an expert legal contract annotator. Given a contract excerpt, your task is to generate a question and an answer in the style of the CUAD dataset."},
            {"role": "user", "content": prompt + el["context"]},
        ]
        outputs = pipe(
            messages,
            max_new_tokens=512,
            num_return_sequences=multiplication_ratio,
        )
        for idx in range(multiplication_ratio):
            new_id = el["title"] + f"__gen_0_{idx}"
            raw_generation = outputs[idx]["generated_text"][-1]["content"]
            try:
                gen_question = raw_generation.split("Question:")[1].split("\n")[0].strip()
            except:
                gen_question = raw_generation
            try:
                gen_answer = raw_generation.split("Answer:")[1].split("\n")[0].strip()
            except:
                gen_answer = raw_generation
            # validate qa
            if gen_answer not in el["context"]:
                print("Answer did not match the context")
                continue
            # select unique questions and add generated samples
            if gen_question not in buf_questions:
                generated_pairs["id"].append(new_id)
                generated_pairs["title"].append(el["title"])
                generated_pairs["context"].append(el["context"])
                generated_pairs["question"].append(gen_question)
                generated_pairs["answers"].append(gen_answer)
            buf_questions.append(gen_question)
    return generated_pairs

In [288]:
# generate more samples for CUAD
# firstly take the dataset train split by length
# after generate more question-answer pairs
# finally validate question-answer pairs by checking if it is in context

In [289]:
dat_2k = datasets.load_from_disk("../../data/cuad_prepared/cuad_no_ns_0k_2k")

In [290]:
dat_2k["train"]

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
    num_rows: 421
})

In [291]:
dat_2k["train"][5]

{'id': 'PfHospitalityGroupInc_20150923_10-12G_EX-10.1_9266710_EX-10.1_Franchise Agreement3__Parties_0',
 'title': 'PfHospitalityGroupInc_20150923_10-12G_EX-10.1_9266710_EX-10.1_Franchise Agreement3',
 'context': 'APPENDIX C SAMPLE OF NON-DISCLOSURE AND NON-COMPETITION AGREEMENT (BETWEEN FRANCHISEE AND ITS PERSONNEL) THIS SAMPLE OF NON-DISCLOSURE AND NON-COMPETITION AGREEMENT ( "Agreement")  is  made this  _____ day of _________, 20___, by and between ___________________________ (the "Franchisee"), and ___________________________, who is an officer, director, or employee of Franchisee (the "Member"). RECITALS: WHEREAS, __________________ ("Franchisor") has developed a distinctive set of specifications and operating procedures (collectively, the "System") for the operation of "Pizza Fusion" restaurant businesses ("Franchised Businesses"). WHEREAS, Franchisor and Franchisee have executed a Franchise Agreement ("Franchise Agreement") granting Franchisee the right to operate a Franchised Bu

In [292]:
generation_prompt = """You are an expert legal contract annotator. Given a contract excerpt, your task is to generate a question and an answer in the style of the CUAD dataset.

    The question should refer to a specific legal clause or concept typically found in business contracts (e.g., "Does this contract contain a change of control clause?", "What is the governing law of this agreement?").

    The answer must be a direct extract (verbatim span) from the contract context.

    Make sure the answer is fully contained within the given context (i.e., do not infer or paraphrase).

    Use concise and legally relevant questions.

    Provide the character start and end indices (inclusive) of the answer span within the context text.

    Return the result in the following format:

    Question: <Generated question>
    Answer: <Exact answer span from context>
    Start: <start_char_index>
    End: <end_char_index>

    Here is the contract excerpt:
    
"""

In [293]:
from transformers import pipeline, set_seed
import torch

In [294]:
model_id = "meta-llama/Llama-3.1-8B-Instruct"
set_seed(42)
pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0


In [295]:
#dat_2k_additional = enlarge_split_by_length(dat_2k, pipe,  min_length=0, max_length=2048, prompt=generation_prompt, multiplication_ratio=2, filter_ns=True)
dat_2k_additional_fix = enlarge_split_by_length(dat_2k, pipe,  min_length=0, max_length=2048, prompt=generation_prompt, multiplication_ratio=30, filter_ns=True)

  4%|▍         | 2/47 [00:12<05:00,  6.68s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Answer did not match the context


  9%|▊         | 4/47 [00:34<07:07,  9.94s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context


 17%|█▋        | 8/47 [00:52<03:22,  5.19s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

 94%|█████████▎| 44/47 [07:14<00:34, 11.45s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Answer did not match the context


 96%|█████████▌| 45/47 [07:19<00:19,  9.66s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Answer did not match the context


100%|██████████| 47/47 [07:31<00:00,  9.61s/it]


In [296]:
len(dat_2k_additional_fix["id"])

401

In [297]:
len(np.unique(dat_2k["train"]["title"]))

47

In [298]:
# now postprocess and save additional dataset
dat_2k_additional_fix["answers"] = [{'text': [el], 'answer_start': [0]} for el in dat_2k_additional_fix["answers"]]
dat_2k_additional_ds = datasets.Dataset.from_dict(dat_2k_additional_fix)
dat_2k_additional_ds = dat_2k_additional_ds.map(tokenize)

Map:   0%|          | 0/401 [00:00<?, ? examples/s]

In [299]:
dat_2k_additional_ds = dat_2k_additional_ds.cast(dat_2k_big["train"].features)

Casting the dataset:   0%|          | 0/401 [00:00<?, ? examples/s]

In [300]:
def filter_duplicates(data, key="title"):
    unique_keys = np.unique(data[key])
    ids_to_remove = []
    for unique_key in unique_keys:
        buf_dat = data.filter(lambda example: example[key] == unique_key)
        unique_questions = []
        for el in buf_dat:
            if el["question"] not in unique_questions:
                unique_questions.append(el["question"])
            else:
                ids_to_remove.append(el["id"])
    filtered_data = data.filter(lambda example: example["id"] not in ids_to_remove)
    return filtered_data

In [301]:
dat_2k_additional_ds_filtered = filter_duplicates(dat_2k_additional_ds)

Filter:   0%|          | 0/401 [00:00<?, ? examples/s]

Filter:   0%|          | 0/401 [00:00<?, ? examples/s]

Filter:   0%|          | 0/401 [00:00<?, ? examples/s]

Filter:   0%|          | 0/401 [00:00<?, ? examples/s]

Filter:   0%|          | 0/401 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/401 [00:00<?, ? examples/s]

Filter:   0%|          | 0/401 [00:00<?, ? examples/s]

Filter:   0%|          | 0/401 [00:00<?, ? examples/s]

Filter:   0%|          | 0/401 [00:00<?, ? examples/s]

Filter:   0%|          | 0/401 [00:00<?, ? examples/s]

In [302]:
dat_2k_additional_ds_filtered

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
    num_rows: 401
})

In [303]:
dat_2k_big = copy.deepcopy(dat_2k)
dat_2k_big["train"] = datasets.concatenate_datasets([dat_2k_big["train"], dat_2k_additional_ds])
dat_2k_big["train"] = dat_2k_big["train"].shuffle(seed=42)

In [304]:
dat_2k_big

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
        num_rows: 822
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
        num_rows: 71
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
        num_rows: 49
    })
})

In [305]:
save_path = f"../../data/cuad_prepared/cuad_no_ns_gen_0k_2k"
dat_2k_big.save_to_disk(save_path)

Saving the dataset (0/1 shards):   0%|          | 0/822 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/71 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/49 [00:00<?, ? examples/s]

## Now for 4s data


In [306]:
# now for 4s data
dat_4k = datasets.load_from_disk("../../data/cuad_prepared/cuad_no_ns_0k_4k")

In [307]:
from transformers import pipeline, set_seed
import torch

In [308]:
model_id = "meta-llama/Llama-3.1-8B-Instruct"
set_seed(42)
pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0


In [310]:
#dat_2k_additional = enlarge_split_by_length(dat_2k, pipe,  min_length=0, max_length=2048, prompt=generation_prompt, multiplication_ratio=2, filter_ns=True)
dat_4k_additional_fix = enlarge_split_by_length(dat_4k, pipe, min_length=2048, max_length=4096, prompt=generation_prompt, multiplication_ratio=30, filter_ns=True)

1410


Filter:   0%|          | 0/1410 [00:00<?, ? examples/s]

Filter:   0%|          | 0/261 [00:00<?, ? examples/s]

Filter:   0%|          | 0/165 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1410 [00:00<?, ? examples/s]

...

Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context


 99%|█████████▊| 66/67 [23:33<00:14, 14.06s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context


100%|██████████| 67/67 [24:03<00:00, 21.54s/it]

Answer did not match the context
Answer did not match the context


In [311]:
len(dat_4k_additional_fix["id"])

239

In [312]:
len(np.unique(dat_4k["train"]["title"]))

111

In [313]:
# now postprocess and save additional dataset
dat_4k_additional_fix["answers"] = [{'text': [el], 'answer_start': [0]} for el in dat_4k_additional_fix["answers"]]
dat_4k_additional_ds = datasets.Dataset.from_dict(dat_4k_additional_fix)
dat_4k_additional_ds = dat_4k_additional_ds.map(tokenize)

Map:   0%|          | 0/239 [00:00<?, ? examples/s]

In [314]:
dat_4k_additional_ds = dat_4k_additional_ds.cast(dat_4k["train"].features)

Casting the dataset:   0%|          | 0/239 [00:00<?, ? examples/s]

In [320]:
dat_4k_additional_ds, dat_2k_additional_ds

(Dataset({
     features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
     num_rows: 239
 }),
 Dataset({
     features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
     num_rows: 401
 }))

In [317]:
dat_4k_big = copy.deepcopy(dat_4k)
dat_4k_big["train"] = datasets.concatenate_datasets([dat_4k_big["train"], dat_4k_additional_ds, dat_2k_additional_ds])
dat_4k_big["train"] = dat_4k_big["train"].shuffle(seed=42)

In [318]:
dat_4k_big

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
        num_rows: 2050
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
        num_rows: 261
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
        num_rows: 165
    })
})

In [321]:
save_path = f"../../data/cuad_prepared/cuad_no_ns_gen_0k_4k"
dat_4k_big.save_to_disk(save_path)

Saving the dataset (0/1 shards):   0%|          | 0/2050 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/261 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/165 [00:00<?, ? examples/s]

# Gen more CUAD samples, 7 models

In [4]:
from tqdm import tqdm
import copy
from transformers import set_seed

In [5]:
# enhance 2k split 
def enlarge_split_by_length(dataset, pipe, min_length=0, max_length=2048, prompt=None, multiplication_ratio=1,
                            filter_ns=False):
    base_seed = 42
    set_seed(base_seed)
    print(len(dataset["train"]))
    if filter_ns:
        buf_data = dataset.filter(lambda example: len(example['answers']['text']) != 0)
        buf_data = buf_data.filter(lambda example: np.logical_and(len(example["tokens"]) >= min_length, len(example["tokens"]) < max_length))
    else:
        # filter samples by length
        buf_data = dataset.filter(lambda example: np.logical_and(len(example["tokens"]) >= min_length, len(example["tokens"]) < max_length))
    # split valdation from train, but check that context does not overlap between train and val
    print(len(buf_data["train"]))
    generated_pairs = {
        'id': [],
        'title': [],
        'context': [],
        'question': [],
        'answers': [],
    }
    # select only unique context
    unique_data_ids = []
    unique_titles = []
    for el in dataset["train"]:
        if el["title"] not in unique_titles:
            unique_titles.append(el["title"])
            unique_data_ids.append(el["id"])
    buf_data = buf_data["train"].filter(lambda example: example["id"] in unique_data_ids)
    print(len(buf_data))
    for el in tqdm(buf_data):
        buf_questions = []
        #while ratio_count < multiplication_ratio and trials < multiplication_ratio * 2:
        #set_seed(base_seed+trials)
        #trials += 1
        # generate qa
        if "gemma-2" in pipe.model.name_or_path:
            # no system role
            messages = [
                {"role": "user", "content": prompt + el["context"]},
            ]

        else:
            messages = [
                {"role": "system", "content": "You are an expert legal contract annotator. Given a contract excerpt, your task is to generate a question and an answer in the style of the CUAD dataset."},
                {"role": "user", "content": prompt + el["context"]},
            ]
        if "Qwen3" in pipe.model.name_or_path:
            outputs = pipe(
                messages,
                max_new_tokens=512,
                num_return_sequences=multiplication_ratio,
                enable_thinking=False,
            )
        elif "Mistral" in pipe.model.name_or_path or "gemma-2" in pipe.model.name_or_path or "Phi" in pipe.model.name_or_path:
            outputs = pipe(
                messages,
                max_new_tokens=512,
                num_return_sequences=multiplication_ratio,
                do_sample=True,
                top_p=0.95,
                top_k=50,
            )
        else:
            outputs = pipe(
                messages,
                max_new_tokens=512,
                num_return_sequences=multiplication_ratio,
            )
        for idx in range(multiplication_ratio):
            new_id = el["title"] + f"__gen_0_{idx}"
            raw_generation = outputs[idx]["generated_text"][-1]["content"]
            # added strip('"') for phi-3.5 generation
            try:
                gen_question = raw_generation.split("Question:")[1].split("\n")[0].strip().strip('"')
            except:
                gen_question = raw_generation
            try:
                gen_answer = raw_generation.split("Answer:")[1].split("\n")[0].strip().strip('"')
            except:
                gen_answer = raw_generation
            # validate qa
            if gen_answer not in el["context"]:
                print("Answer did not match the context")
                continue
            # select unique questions and add generated samples
            if gen_question not in buf_questions:
                generated_pairs["id"].append(new_id)
                generated_pairs["title"].append(el["title"])
                generated_pairs["context"].append(el["context"])
                generated_pairs["question"].append(gen_question)
                generated_pairs["answers"].append(gen_answer)
            buf_questions.append(gen_question)
    return generated_pairs

In [6]:
dat_2k = datasets.load_from_disk("../../data/cuad_prepared/cuad_no_ns_0k_2k")

In [7]:
dat_2k["train"]

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
    num_rows: 421
})

In [12]:
generation_prompt = """You are an expert legal contract annotator. Given a contract excerpt, your task is to generate a question and an answer in the style of the CUAD dataset.

    The question should refer to a specific legal clause or concept typically found in business contracts (e.g., "Does this contract contain a change of control clause?", "What is the governing law of this agreement?").

    The answer must be a direct extract (verbatim span) from the contract context.

    Make sure the answer is fully contained within the given context (i.e., do not infer or paraphrase).

    Use concise and legally relevant questions.

    Provide the character start and end indices (inclusive) of the answer span within the context text.

    Return the result in the following format:

    Question: <Generated question>
    Answer: <Exact answer span from context>
    Start: <start_char_index>
    End: <end_char_index>

    Here is the contract excerpt:
    
"""

In [13]:
from transformers import pipeline, set_seed
import torch
import gc
import time

In [10]:
torch.cuda.current_device()

0

In [11]:
print("torch.cuda.memory_allocated: %fGB"%(torch.cuda.memory_allocated(0)/1024/1024/1024))
print("torch.cuda.memory_reserved: %fGB"%(torch.cuda.memory_reserved(0)/1024/1024/1024))
print("torch.cuda.max_memory_reserved: %fGB"%(torch.cuda.max_memory_reserved(0)/1024/1024/1024))

torch.cuda.memory_allocated: 0.000000GB
torch.cuda.memory_reserved: 0.000000GB
torch.cuda.max_memory_reserved: 0.000000GB


In [12]:
print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |

In [13]:
# iterate over several models
models = [
    "meta-llama/Llama-3.1-8B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "Qwen/Qwen2.5-7B-Instruct",
    "google/gemma-3-4b-it",
    "google/gemma-2-9b-it",
    "microsoft/Phi-4-mini-instruct",
    "microsoft/Phi-3.5-mini-instruct",
]
gen_data = []
min_len = 0
max_len = 2048
mult_ratio = 5
for model_id in models:
    set_seed(42)
    pipe = pipeline(
        "text-generation",
        model=model_id,
        torch_dtype=torch.bfloat16,
        device_map="cuda",
    )
    print("Model loaded")
    dat_2k_additional_fix = enlarge_split_by_length(dat_2k, pipe,  min_length=min_len, max_length=max_len,
                                                    prompt=generation_prompt, multiplication_ratio=mult_ratio,
                                                    filter_ns=True)
    print(f"Num of gen samples for model {model_id}: {len(dat_2k_additional_fix['id'])}")
    gen_data.append(dat_2k_additional_fix)
    del pipe
    torch.cuda.empty_cache()
    gc.collect()
    time.sleep(30)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda


Model loaded
421
421
47


  9%|▊         | 4/47 [00:11<01:55,  2.68s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Answer did not match the context
Answer did not match the context


...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda


Model loaded
421
421
47


  0%|          | 0/47 [00:00<?, ?it/s]


ValueError: Greedy methods without beam search do not support `num_return_sequences` different than 1 (got 5).

In [15]:
del pipe
torch.cuda.empty_cache()
gc.collect()
time.sleep(30)

In [16]:
# iterate over several models
models = [
    "microsoft/Phi-4-mini-instruct",
    "microsoft/Phi-3.5-mini-instruct",
]
min_len = 0
max_len = 2048
mult_ratio = 5
for model_id in models:
    set_seed(42)
    pipe = pipeline(
        "text-generation",
        model=model_id,
        torch_dtype=torch.bfloat16,
        device_map="cuda",
    )
    print("Model loaded")
    dat_2k_additional_fix = enlarge_split_by_length(dat_2k, pipe,  min_length=min_len, max_length=max_len,
                                                    prompt=generation_prompt, multiplication_ratio=mult_ratio,
                                                    filter_ns=True)
    print(f"Num of gen samples for model {model_id}: {len(dat_2k_additional_fix['id'])}")
    gen_data.append(dat_2k_additional_fix)
    del pipe
    torch.cuda.empty_cache()
    gc.collect()
    time.sleep(30)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda


Model loaded
421
421
47


  2%|▏         | 1/47 [00:07<05:25,  7.07s/it]

Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context


...

Answer did not match the context


 98%|█████████▊| 46/47 [07:33<00:09,  9.68s/it]

Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context


100%|██████████| 47/47 [07:38<00:00,  9.75s/it]


Num of gen samples for model microsoft/Phi-3.5-mini-instruct: 180


In [28]:
# iterate over several models
models = [
    "microsoft/Phi-3.5-mini-instruct",
]
min_len = 0
max_len = 2048
mult_ratio = 5
for model_id in models:
    set_seed(42)
    pipe = pipeline(
        "text-generation",
        model=model_id,
        torch_dtype=torch.bfloat16,
        device_map="cuda",
    )
    print("Model loaded")
    dat_2k_additional_fix = enlarge_split_by_length(dat_2k, pipe,  min_length=min_len, max_length=max_len,
                                                    prompt=generation_prompt, multiplication_ratio=mult_ratio,
                                                    filter_ns=True)
    print(f"Num of gen samples for model {model_id}: {len(dat_2k_additional_fix['id'])}")
    gen_data[-1] = dat_2k_additional_fix
    del pipe
    torch.cuda.empty_cache()
    gc.collect()
    time.sleep(30)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda


Model loaded
421
421
47


  2%|▏         | 1/47 [00:14<10:46, 14.05s/it]

Answer did not match the context


...

Answer did not match the context


 98%|█████████▊| 46/47 [07:33<00:09,  9.65s/it]

Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context


100%|██████████| 47/47 [07:38<00:00,  9.75s/it]


Num of gen samples for model microsoft/Phi-3.5-mini-instruct: 180


In [7]:
tokenizer = AutoTokenizer.from_pretrained("unsloth/Llama-3.2-1B-Instruct")

In [8]:
# tokenize texts to calc length
def tokenize(el):
    if len(el['answers']['text']) != 0:
        answer = el['answers']['text'][0]
    else:
        answer = "Not stated."
    messages = [
        {"role": "user", "content": f"Question:\n{el['question']}\nContext:\n{el['context']}\n"},
        {"role": "assistant", "content": f"Answer:\n{answer}\n"},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )
    return {"tokens": text}

In [32]:
# now postprocess and save additional dataset
new_gen_data = []
for buf_data in gen_data:
    dat_2k_additional_fix_1 = copy.deepcopy(buf_data)
    dat_2k_additional_fix_1["answers"] = [{'text': [str(el)], 'answer_start': [0]} for el in buf_data["answers"]]
    dat_2k_additional_ds = datasets.Dataset.from_dict(dat_2k_additional_fix_1)
    dat_2k_additional_ds = dat_2k_additional_ds.map(tokenize)
    dat_2k_additional_ds = dat_2k_additional_ds.cast(dat_2k["train"].features)
    new_gen_data.append(dat_2k_additional_ds)

Map:   0%|          | 0/125 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/125 [00:00<?, ? examples/s]

Map:   0%|          | 0/145 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/145 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

...

Casting the dataset:   0%|          | 0/114 [00:00<?, ? examples/s]

Map:   0%|          | 0/138 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/138 [00:00<?, ? examples/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/180 [00:00<?, ? examples/s]

In [33]:
new_gen_data_7_5 = datasets.concatenate_datasets(new_gen_data)
new_gen_data_7_5.save_to_disk("../../data/cuad_prepared/cuad_raw_no_ns_gen_only_7_5_0k_2k")

Saving the dataset (0/1 shards):   0%|          | 0/894 [00:00<?, ? examples/s]

In [34]:
def filter_duplicates(data, key="title"):
    unique_keys = np.unique(data[key])
    ids_to_remove = []
    for unique_key in unique_keys:
        buf_dat = data.filter(lambda example: example[key] == unique_key)
        unique_questions = []
        for el in buf_dat:
            if el["question"] not in unique_questions:
                unique_questions.append(el["question"])
            else:
                ids_to_remove.append(el["id"])
    filtered_data = data.filter(lambda example: example["id"] not in ids_to_remove)
    return filtered_data

In [35]:
dat_2k_additional_ds_filtered = filter_duplicates(new_gen_data_7_5)

Filter:   0%|          | 0/894 [00:00<?, ? examples/s]

Filter:   0%|          | 0/894 [00:00<?, ? examples/s]

Filter:   0%|          | 0/894 [00:00<?, ? examples/s]

Filter:   0%|          | 0/894 [00:00<?, ? examples/s]

Filter:   0%|          | 0/894 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/894 [00:00<?, ? examples/s]

Filter:   0%|          | 0/894 [00:00<?, ? examples/s]

Filter:   0%|          | 0/894 [00:00<?, ? examples/s]

Filter:   0%|          | 0/894 [00:00<?, ? examples/s]

Filter:   0%|          | 0/894 [00:00<?, ? examples/s]

In [36]:
dat_2k_additional_ds_filtered

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
    num_rows: 601
})

In [41]:
dat_2k_big = copy.deepcopy(dat_2k)
dat_2k_big["train"] = datasets.concatenate_datasets([dat_2k_big["train"], dat_2k_additional_ds_filtered])
dat_2k_big["train"] = dat_2k_big["train"].shuffle(seed=42)

In [42]:
dat_2k_big

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
        num_rows: 1022
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
        num_rows: 71
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
        num_rows: 49
    })
})

In [43]:
save_path = f"../../data/cuad_prepared/cuad_no_ns_gen_7_5_0k_2k"
dat_2k_big.save_to_disk(save_path)

Saving the dataset (0/1 shards):   0%|          | 0/1022 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/71 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/49 [00:00<?, ? examples/s]

## Now for 4s data


In [9]:
# now for 4s data
dat_4k = datasets.load_from_disk("../../data/cuad_prepared/cuad_no_ns_0k_4k")

In [10]:
from transformers import pipeline, set_seed
import torch

[2025-06-26 22:16:18,811] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/jovyan/gkuzmin/rmt_it/conda_rmt_it_venv/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/jovyan/gkuzmin/rmt_it/conda_rmt_it_venv/compiler_compat/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status


[2025-06-26 22:16:22,128] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False


In [14]:
# iterate over several models
models = [
    "meta-llama/Llama-3.1-8B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "Qwen/Qwen2.5-7B-Instruct",
    "google/gemma-3-4b-it",
    "google/gemma-2-9b-it",
    "microsoft/Phi-4-mini-instruct",
    "microsoft/Phi-3.5-mini-instruct",
]
gen_data = []
min_len = 2048
max_len = 4096
mult_ratio = 5
for model_id in models:
    set_seed(42)
    pipe = pipeline(
        "text-generation",
        model=model_id,
        torch_dtype=torch.bfloat16,
        device_map="cuda",
    )
    print("Model loaded")
    dat_4k_additional_fix = enlarge_split_by_length(dat_4k, pipe,  min_length=min_len, max_length=max_len,
                                                    prompt=generation_prompt, multiplication_ratio=mult_ratio,
                                                    filter_ns=True)
    print(f"Num of gen samples for model {model_id}: {len(dat_4k_additional_fix['id'])}")
    gen_data.append(dat_4k_additional_fix)
    del pipe
    torch.cuda.empty_cache()
    gc.collect()
    time.sleep(30)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda


Model loaded
1410
1018
67


  4%|▍         | 3/67 [00:07<02:35,  2.43s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Answer did not match the context
Answer did not match the context
Answer did not match the context
Answer did not match the context


...

Answer did not match the context
Answer did not match the context
Answer did not match the context


  4%|▍         | 3/67 [00:37<12:25, 11.65s/it]

Answer did not match the context
Answer did not match the context
Answer did not match the context


  4%|▍         | 3/67 [00:37<13:23, 12.55s/it]


RuntimeError: p.attn_bias_ptr is not correctly aligned

In [ ]:
del pipe
torch.cuda.empty_cache()
gc.collect()
time.sleep(30)

In [ ]:
models = [
    #"google/gemma-3-4b-it",
    "google/gemma-2-9b-it",
    "microsoft/Phi-4-mini-instruct",
    "microsoft/Phi-3.5-mini-instruct",
]
min_len = 2048
max_len = 4096
mult_ratio = 5
for model_id in models:
    set_seed(42)
    pipe = pipeline(
        "text-generation",
        model=model_id,
        torch_dtype=torch.bfloat16,
        device_map="cuda",
    )
    print("Model loaded")
    dat_4k_additional_fix = enlarge_split_by_length(dat_4k, pipe,  min_length=min_len, max_length=max_len,
                                                    prompt=generation_prompt, multiplication_ratio=mult_ratio,
                                                    filter_ns=True)
    print(f"Num of gen samples for model {model_id}: {len(dat_4k_additional_fix['id'])}")
    gen_data.append(dat_4k_additional_fix)
    del pipe
    torch.cuda.empty_cache()
    gc.collect()
    time.sleep(30)

In [ ]:
# now postprocess and save additional dataset
new_gen_data = []
for buf_data in gen_data:
    dat_4k_additional_fix_1 = copy.deepcopy(buf_data)
    dat_4k_additional_fix_1["answers"] = [{'text': [str(el)], 'answer_start': [0]} for el in buf_data["answers"]]
    dat_4k_additional_ds = datasets.Dataset.from_dict(dat_4k_additional_fix_1)
    dat_4k_additional_ds = dat_4k_additional_ds.map(tokenize)
    dat_4k_additional_ds = dat_4k_additional_ds.cast(dat_4k["train"].features)
    new_gen_data.append(dat_4k_additional_ds)

In [ ]:
new_gen_data_7_5_4k = datasets.concatenate_datasets(new_gen_data)
new_gen_data_7_5_4k.save_to_disk("../../data/cuad_prepared/cuad_raw_no_ns_gen_only_7_5_0k_4k")

In [22]:
#new_gen_data_7_5_4k = datasets.load_from_disk("../../data/cuad_prepared/cuad_raw_no_ns_gen_only_7_5_0k_4k")

In [23]:
def filter_duplicates(data, key="title"):
    unique_keys = np.unique(data[key])
    ids_to_remove = []
    for unique_key in unique_keys:
        buf_dat = data.filter(lambda example: example[key] == unique_key)
        unique_questions = []
        for el in buf_dat:
            if el["question"] not in unique_questions:
                unique_questions.append(el["question"])
            else:
                ids_to_remove.append(el["id"])
    filtered_data = data.filter(lambda example: example["id"] not in ids_to_remove)
    return filtered_data

In [24]:
dat_4k_additional_ds_filtered = filter_duplicates(new_gen_data_7_5_4k)

Filter:   0%|          | 0/660 [00:00<?, ? examples/s]

Filter:   0%|          | 0/660 [00:00<?, ? examples/s]

Filter:   0%|          | 0/660 [00:00<?, ? examples/s]

Filter:   0%|          | 0/660 [00:00<?, ? examples/s]

Filter:   0%|          | 0/660 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/660 [00:00<?, ? examples/s]

Filter:   0%|          | 0/660 [00:00<?, ? examples/s]

Filter:   0%|          | 0/660 [00:00<?, ? examples/s]

Filter:   0%|          | 0/660 [00:00<?, ? examples/s]

Filter:   0%|          | 0/660 [00:00<?, ? examples/s]

In [25]:
dat_4k_additional_ds_filtered

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
    num_rows: 437
})

In [26]:
# load and extract dat_2k_additional_ds_filtered
new_gen_data_7_5_2k = datasets.load_from_disk("../../data/cuad_prepared/cuad_raw_no_ns_gen_only_7_5_0k_2k")

In [27]:
dat_2k_additional_ds_filtered = filter_duplicates(new_gen_data_7_5_2k)

Filter:   0%|          | 0/894 [00:00<?, ? examples/s]

Filter:   0%|          | 0/894 [00:00<?, ? examples/s]

Filter:   0%|          | 0/894 [00:00<?, ? examples/s]

Filter:   0%|          | 0/894 [00:00<?, ? examples/s]

Filter:   0%|          | 0/894 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/894 [00:00<?, ? examples/s]

Filter:   0%|          | 0/894 [00:00<?, ? examples/s]

Filter:   0%|          | 0/894 [00:00<?, ? examples/s]

Filter:   0%|          | 0/894 [00:00<?, ? examples/s]

Filter:   0%|          | 0/894 [00:00<?, ? examples/s]

In [30]:
dat_4k_additional_ds_filtered, dat_2k_additional_ds_filtered, dat_4k

(Dataset({
     features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
     num_rows: 437
 }),
 Dataset({
     features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
     num_rows: 601
 }),
 DatasetDict({
     train: Dataset({
         features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
         num_rows: 1410
     })
     test: Dataset({
         features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
         num_rows: 261
     })
     validation: Dataset({
         features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
         num_rows: 165
     })
 }))

In [28]:
dat_4k_big = copy.deepcopy(dat_4k)
dat_4k_big["train"] = datasets.concatenate_datasets([dat_4k_big["train"], dat_4k_additional_ds_filtered, dat_2k_additional_ds_filtered])
dat_4k_big["train"] = dat_4k_big["train"].shuffle(seed=42)

In [29]:
dat_4k_big

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
        num_rows: 2448
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
        num_rows: 261
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
        num_rows: 165
    })
})

In [31]:
save_path = f"../../data/cuad_prepared/cuad_no_ns_gen_7_5_0k_4k"
dat_4k_big.save_to_disk(save_path)

Saving the dataset (0/1 shards):   0%|          | 0/2448 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/261 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/165 [00:00<?, ? examples/s]

In [ ]:
# now postprocess and save additional dataset
dat_4k_additional_fix["answers"] = [{'text': [el], 'answer_start': [0]} for el in dat_4k_additional_fix["answers"]]
dat_4k_additional_ds = datasets.Dataset.from_dict(dat_4k_additional_fix)
dat_4k_additional_ds = dat_4k_additional_ds.map(tokenize)

Map:   0%|          | 0/239 [00:00<?, ? examples/s]

In [ ]:
dat_4k_additional_ds = dat_4k_additional_ds.cast(dat_4k["train"].features)

Casting the dataset:   0%|          | 0/239 [00:00<?, ? examples/s]

In [ ]:
dat_4k_additional_ds, dat_2k_additional_ds

(Dataset({
     features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
     num_rows: 239
 }),
 Dataset({
     features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
     num_rows: 401
 }))

In [ ]:
dat_4k_big = copy.deepcopy(dat_4k)
dat_4k_big["train"] = datasets.concatenate_datasets([dat_4k_big["train"], dat_4k_additional_ds, dat_2k_additional_ds])
dat_4k_big["train"] = dat_4k_big["train"].shuffle(seed=42)

In [ ]:
dat_4k_big

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
        num_rows: 2050
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
        num_rows: 261
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'tokens'],
        num_rows: 165
    })
})

In [ ]:
save_path = f"../../data/cuad_prepared/cuad_no_ns_gen_0k_4k"
dat_4k_big.save_to_disk(save_path)

Saving the dataset (0/1 shards):   0%|          | 0/2050 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/261 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/165 [00:00<?, ? examples/s]

In [323]:
import gc
del pipe
torch.cuda.empty_cache()
gc.collect()

NameError: name 'pipe' is not defined

In [223]:
# generate qa
messages = [
    {"role": "system", "content": "You are an expert legal contract annotator. Given a contract excerpt, your task is to generate a question and an answer in the style of the CUAD dataset."},
    {"role": "user", "content": generation_prompt + dat_2k["train"][4]["context"]},
]
outputs = pipe(
    messages,
    max_new_tokens=512,
    num_return_sequences=10
)
raw_generation = outputs[0]["generated_text"][-1]["content"]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [224]:
raw_generation

'Question: What is the definition of "Competitive Business" in this non-disclosure and non-competition agreement?\nAnswer: A restaurant featuring pizza and related food specialties or that offers products or services substantially similar to those then offered by Pizza Fusions Restaurants.\nStart: 121\nEnd: 158'

In [241]:
outputs[5]["generated_text"][-1]["content"]

'Question: What is the definition of "Confidential Information" in this agreement?\nAnswer: All information, knowledge and know-how relating to Franchisor, its business plans, Franchised Businesses, or the System ("Confidential Information") is deemed confidential, except for information that Member can demonstrate came to Member\'s attention by lawful means prior to disclosure to Member; or which, at the time of the disclosure to Member, had become a part of the public domain.\nStart: 145\nEnd: 216'

In [110]:
raw_generation.split("Answer:")[1].split("\n")[0].strip()

"this Schedule 13G or Schedule 13D and any future amendments thereto reporting each of the undersigned's ownership of securities of the issuer named herein"

In [ ]:
dat_2k_big = enlarge_split_by_length(dat_2k, min_length=0, max_length=2048, prompt=generation_prompt, filter_ns=True)

In [53]:
un_context = set(tok_cuad["train"]["context"])

In [54]:
len(un_context)

407

In [55]:
un_context_32 = set(buf_data["train"]["context"])

In [56]:
len(un_context_32)

341

In [99]:
val, counts = np.unique(b1["train"]["context"], return_counts=True)

In [100]:
counts

array([45, 42, 44, 44, 44, 45, 46, 46, 45, 45, 44, 46, 44, 41, 45, 46, 45,
       44, 45, 42, 45, 51, 44, 44, 44, 46, 45, 42, 45, 44, 38, 48, 45, 44,
       50, 43, 50, 44, 43, 45, 43, 38, 44, 45, 44, 44, 44, 44, 45, 47, 44,
       44])

In [ ]:
len(counts)

In [ ]:
_, counts_all = np.unique(tok_cuad["train"]["context"], return_counts=True)

In [ ]:
len(counts_all)

In [ ]:
counts_all

In [36]:
tokens = tokenizer(cuad["train"]["context"])

In [38]:
lengths = [len(el) for el in tokens["input_ids"]]

In [52]:
np.min(lengths), np.max(lengths), np.mean(lengths), np.percentile(lengths,20), np.percentile(lengths,80), np.percentile(lengths,90)

(np.int64(235),
 np.int64(80233),
 np.float64(13408.608062360801),
 np.float64(3375.0),
 np.float64(22305.0),
 np.float64(32339.0))

In [63]:
np.sum(np.array(lengths) < 2048), np.sum(np.logical_and(np.array(lengths) < 4096, np.array(lengths) > 2048)),\
np.sum(np.logical_and(np.array(lengths) < 8192, np.array(lengths) > 4096)),\
np.sum(np.logical_and(np.array(lengths) < 16384, np.array(lengths) > 8192)),\
np.sum(np.logical_and(np.array(lengths) < 32768, np.array(lengths) > 16384))

(np.int64(2383),
 np.int64(3419),
 np.int64(4897),
 np.int64(5771),
 np.int64(3877))

In [45]:
cuad["train"][2]

{'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Parties_1',
 'title': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT',
 'context': 'EXHIBIT 10.6\n\n                              DISTRIBUTOR AGREEMENT\n\n         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.\n\n                                    RECITALS\n\n         A. The  Company\'s  Business.  The Company is  presently  engaged in the business  of selling an energy  efficiency  device,  which is  referred to as an "Energy  Saver"  which may be improved  or  otherwise  changed  from its present composition (the "Products").  The Company may engage in the business of selling other  products  or  other  devices  other  than  the  Products,  which  will be considered  Products if Distributor  exercises its options pursuant to Section 7 hereof.\n\n     